# Confirmed-restatement MD&A study — Colab GPU

This notebook tests whether MD&A predicts a later **issuer-level financial-statement restatement** after an original 10-K/10-Q. It does not use the amendment proxy as the target, and the event need not restate the exact period of the original filing. It evaluates a fixed 1:1 case-control cohort and reports development and forward-year holdout metrics. No score is promised in advance.

The ledger may contain human-adjudicated labels or an AI-assisted SEC evidence screen. AI-assisted labels are exploratory; the validator checks structure and source links, not accounting correctness. Do not report confirmatory or human-validated performance until outcomes have been human-audited. Pending rows are never used.

Choose one outcome horizon before the run: 1, 3, or 5 years. A negative needs documented SEC filing review through the end of that horizon. A confirmed event date is retained, so targets at other horizons can be derived later. SEC text is downloaded only for reviewed originals and their immediately prior same-issuer/form filing. Downloads are sequential, limited to two requests per second, and stop on access denial. Use a real identifying SEC User-Agent.

The next version adds strictly point-in-time financial trajectories from up to three earlier same-issuer/same-form filings, and sentence embeddings now cover the full available MD&A section (up to 6,000 words) instead of only its opening portion. It compares structured, interpretable-text, semantic-text, and fused features using GPU XGBoost/CatBoost. Training retains every development row using class-weighted loss; validation and the primary holdout remain unique, year/form-matched 1:1 samples. Model selection uses issuer-grouped development PR-AUC, and the threshold is set from development predictions only. The latest filing years stay untouched until final evaluation. The returned 1:1 metrics apply only to this reviewed case-control sample; they do not establish performance at natural prevalence.

In [ ]:
%pip -q uninstall -y xgboost xgboost-cu12
%pip -q install "pandas>=2.2,<3" "numpy>=1.26,<3" "scikit-learn>=1.5,<2" "xgboost-cu12>=3.2,<4" "catboost>=1.2,<2" "sentence-transformers>=3,<6" "beautifulsoup4>=4.12,<5"

If Colab asks for a runtime restart after installation, restart, reselect GPU, and run all cells from the top.

In [ ]:
import torch, subprocess, xgboost as xgb
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU, then rerun."
assert xgb.build_info().get("USE_CUDA",False), "Installed XGBoost lacks CUDA; reinstall dependencies and restart the GPU runtime."
print("CUDA:",torch.version.cuda,"GPU:",torch.cuda.get_device_name(0),"XGBoost CUDA:",xgb.build_info().get("CUDA_VERSION"))
subprocess.run(["nvidia-smi"],check=True)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
ROOT=Path("/content/drive/MyDrive/financial-risk-detector")
PROXY=ROOT/"data/colab_amendment_proxy.csv.gz"
LEDGER=ROOT/"data/restatement_review_queue.csv"
assert PROXY.is_file(), f"Upload the prepared feature CSV: {PROXY}"
assert LEDGER.is_file(), f"Upload the selected review ledger as restatement_review_queue.csv: {LEDGER}"
HORIZON_YEARS=3  # Choose exactly one: 1, 3, or 5
assert HORIZON_YEARS in (1,3,5)
OUTPUT=ROOT/f"colab_outputs/confirmed_restatement_{HORIZON_YEARS}y_fulltext_trajectory_v2"
CACHE=ROOT/f"evidence_restatement/{HORIZON_YEARS}y/sections.sqlite"
print("Proxy:",PROXY,"\nLedger:",LEDGER,"\nHorizon:",HORIZON_YEARS,"years")

In [ ]:
from pathlib import Path
import sys, importlib, importlib.util
MODULES=Path("/content/restatement_modules"); MODULES.mkdir(parents=True,exist_ok=True)
SOURCES={'research_v5_core.py': '"""Point-in-time features and fixed study design; no model fitting or networking."""\nimport hashlib\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nTARGET = \'y_amendment_proxy\'\nSEEDS = (42, 143, 244)\nFOLDS = ((2018, 2020), (2020, 2022))\nRATIOS = (\'accruals_to_assets\', \'net_margin\', \'gross_margin\', \'debt_to_assets\',\n          \'cash_to_assets\', \'working_capital_to_assets\', \'receivables_to_revenue\',\n          \'inventory_to_revenue\', \'operating_cashflow_to_assets\', \'revenue_to_assets\',\n          \'liabilities_to_assets\', \'retained_earnings_to_assets\', \'filing_delay_days\')\nEXCLUDED = {\'adsh\', \'cik\', \'period\', \'period_date\', \'filed\', \'filed_date\', \'filed_year\',\n            \'sic\', \'afs\', \'wksi\', \'days_to_amendment\', TARGET, \'feature_schema_version\',\n            \'label_observation_cutoff\'}\n\n\ndef file_hash(path):\n    result = hashlib.sha256()\n    with open(path, \'rb\') as handle:\n        for chunk in iter(lambda: handle.read(2**20), b\'\'):\n            result.update(chunk)\n    return result.hexdigest()\n\n\ndef load_data(path):\n    df = pd.read_csv(path, dtype={\'adsh\':str, \'cik\':str}, low_memory=False)\n    required = EXCLUDED | {\'form\', \'fp\', \'sic2\', \'prior_amendment_count\'}\n    if required - set(df):\n        raise ValueError(f\'Missing columns: {sorted(required-set(df))}\')\n    if not df.feature_schema_version.eq(4).all():\n        raise ValueError(\'Use the prepared schema-v4 CSV, not an earlier export.\')\n    df[\'filed_date\'] = pd.to_datetime(df.filed_date, errors=\'raise\')\n    df[\'period_date\'] = pd.to_datetime(df.period_date, errors=\'raise\')\n    df[\'cik\'] = df.cik.astype(str).str.lstrip(\'0\').replace(\'\', \'0\')\n    if not df.adsh.is_unique or df[TARGET].isna().any() or not set(df[TARGET]) <= {0,1}:\n        raise ValueError(\'Invalid labels or duplicate accessions.\')\n    cutoff = pd.to_datetime(df.label_observation_cutoff)\n    if cutoff.nunique() != 1 or cutoff.iloc[0] < pd.Timestamp(\'2026-09-30\'):\n        raise ValueError(\'Require label observation through 2026-09-30.\')\n    if not ((cutoff-df.filed_date).dt.days >= 365).all():\n        raise ValueError(\'An outcome has less than 365 days of follow-up.\')\n    if not set(df.form) <= {\'10-K\',\'10-Q\'}:\n        raise ValueError(\'Predictors must come from original 10-K/10-Q filings.\')\n    df[\'sic2\'] = df.sic2.fillna(\'unknown\').astype(str)\n    return df.reset_index(drop=True)\n\n\ndef balanced(part, seed=42):\n    pos, neg = part[part[TARGET].eq(1)], part[part[TARGET].eq(0)]\n    if pos.empty:\n        raise ValueError(\'Cohort has zero positives.\')\n    controls = []\n    for (year, form), group in pos.groupby([\'filed_year\',\'form\']):\n        pool = neg[neg.filed_year.eq(year) & neg.form.eq(form)]\n        if len(pool) < len(group):\n            raise ValueError(f\'Insufficient controls: {year}, {form}\')\n        controls.append(pool.sample(len(group), replace=False, random_state=seed))\n    return pd.concat([pos, *controls]).sample(frac=1, random_state=seed)\n\n\ndef date_part(df, start, end):\n    return df[df.filed_date.between(pd.Timestamp(start),pd.Timestamp(end))]\n\n\ndef training_pool(df, fit_year, window=8):\n    return date_part(df, f\'{max(2009,fit_year-window+1)}-01-01\', f\'{fit_year}-12-31\')\n\n\ndef add_mature_history(df):\n    """A prior label is usable only AFTER its complete 365-day outcome window."""\n    result = pd.DataFrame(index=df.index)\n    for group_cols, prefix in [([\'cik\'],\'issuer\'), ([\'cik\',\'form\'],\'issuer_form\')]:\n        count = np.zeros(len(df)); positive = np.zeros(len(df))\n        for _, group in df.groupby(group_cols, sort=False):\n            history = group.sort_values(\'filed_date\')\n            maturity = (history.filed_date + pd.Timedelta(days=365)).to_numpy(\'datetime64[D]\')\n            at = np.searchsorted(maturity, group.filed_date.to_numpy(\'datetime64[D]\'), side=\'left\')\n            prefix_sum = np.r_[0, history[TARGET].to_numpy().cumsum()]\n            count[group.index] = at\n            positive[group.index] = prefix_sum[at]\n        result[f\'{prefix}_mature_count\'] = count\n        result[f\'{prefix}_mature_positive_count\'] = positive\n        # Fixed smoothing constant; never estimated from later outcome labels.\n        result[f\'{prefix}_mature_rate\'] = (positive + 0.4) / (count + 20)\n    return pd.concat([df, result], axis=1)\n\n\ndef add_financial_history(df):\n    columns = [c for c in RATIOS if c in df]\n    # One aggregate per issuer/form/day prevents same-day information leakage.\n    keys = [\'cik\',\'form\',\'filed_date\']\n    days = df.groupby(keys, as_index=False)[columns].mean().sort_values(keys)\n    groups = days.groupby([\'cik\',\'form\'], sort=False)\n    # Compute all ratios together; per-issuer Python callbacks are very slow\n    # across hundreds of thousands of filings on a standard Colab runtime.\n    past = groups[columns].shift()\n    rolling = past.groupby([days.cik,days.form],sort=False).rolling(3,min_periods=1)\n    means = rolling.mean().reset_index(level=[0,1],drop=True)\n    stds = rolling.std().reset_index(level=[0,1],drop=True)\n    derived = []\n    for col in columns:\n        days[f\'past3_mean_{col}\'] = means[col]\n        days[f\'past3_std_{col}\'] = stds[col]\n        derived += [f\'past3_mean_{col}\', f\'past3_std_{col}\']\n    out = df.merge(days[keys+derived], on=keys, how=\'left\', validate=\'many_to_one\')\n    extra = {}\n    for col in columns:\n        extra[f\'deviation_past3_{col}\'] = out[col]-out[f\'past3_mean_{col}\']\n    out = pd.concat([out, pd.DataFrame(extra,index=out.index)], axis=1)\n    # Calendar-year aggregates come solely from the PREVIOUS filing year.\n    for col in columns:\n        stats = df.groupby([\'filed_year\',\'form\',\'sic2\'])[col].agg([\'median\',\'count\']).reset_index()\n        stats[\'filed_year\'] += 1\n        stats.loc[stats[\'count\'] < 20, \'median\'] = np.nan\n        name = f\'prior_year_peer_median_{col}\'\n        stats = stats.rename(columns={\'median\':name})\n        out = out.merge(stats[[\'filed_year\',\'form\',\'sic2\',name]],\n                        on=[\'filed_year\',\'form\',\'sic2\'], how=\'left\', validate=\'many_to_one\')\n        out[f\'peer_deviation_{col}\'] = out[col]-out[name]\n    out[\'loss_with_positive_cashflow\'] = ((out.net_income_to_assets < 0) &\n                                           (out.operating_cashflow_to_assets > 0)).astype(\'int8\')\n    out[\'profit_with_negative_cashflow\'] = ((out.net_income_to_assets > 0) &\n                                            (out.operating_cashflow_to_assets < 0)).astype(\'int8\')\n    out[\'negative_equity\'] = out.xbrl_StockholdersEquity.lt(0).astype(\'int8\')\n    facts = [c for c in df if c.startswith(\'xbrl_\')]\n    out[\'financial_fact_missing_fraction\'] = out[facts].isna().mean(axis=1)\n    return out\n\n\ndef add_event_history(df, events):\n    """Public EDGAR index events, all strictly before the predictor date."""\n    out = pd.DataFrame(index=df.index)\n    events = events.copy()\n    events[\'filed_date\'] = pd.to_datetime(events.filed_date)\n    events[\'cik\'] = events.cik.astype(str).str.lstrip(\'0\').replace(\'\',\'0\')\n    categories = {\'8k\':[\'8-K\',\'8-K/A\'], \'late\':[\'NT 10-K\',\'NT 10-Q\'],\n                  \'10ka\':[\'10-K/A\'], \'10qa\':[\'10-Q/A\']}\n    for name, forms in categories.items():\n        event_groups = {cik:np.sort(g.filed_date.to_numpy(\'datetime64[D]\'))\n                        for cik,g in events[events.form.isin(forms)].groupby(\'cik\')}\n        counts = {n:np.zeros(len(df), dtype=\'int32\') for n in (90,365,730)}\n        gap = np.full(len(df), np.nan)\n        for cik, rows in df.groupby(\'cik\',sort=False):\n            dates = event_groups.get(cik, np.array([],dtype=\'datetime64[D]\'))\n            query = rows.filed_date.to_numpy(\'datetime64[D]\')\n            end = np.searchsorted(dates,query,side=\'left\')\n            for span in counts:\n                counts[span][rows.index] = end-np.searchsorted(dates,query-np.timedelta64(span,\'D\'),side=\'left\')\n            valid = end > 0\n            gap[rows.index[valid]] = (query[valid]-dates[end[valid]-1]).astype(\'timedelta64[D]\').astype(int)\n        for span,values in counts.items():\n            out[f\'sec_{name}_count_{span}d\'] = values\n        out[f\'sec_{name}_days_since\'] = gap\n    return pd.concat([df,out],axis=1)\n\n\ndef threshold_choice(y, probability):\n    """Maximise the worst of precision/recall/F1/accuracy on CALIBRATION only."""\n    y = np.asarray(y, dtype=int); p = np.asarray(probability, dtype=float)\n    if len(y) == 0 or len(np.unique(y)) < 2 or not np.isfinite(p).all():\n        raise ValueError(\'Threshold selection requires finite scores and both classes.\')\n    order = np.argsort(-p, kind=\'stable\'); ys = y[order]; ps = p[order]\n    ends = np.r_[np.flatnonzero(ps[:-1] != ps[1:]),len(ps)-1]\n    tp = ys.cumsum()[ends]; fp = ends+1-tp\n    recall = tp/y.sum(); precision = tp/(ends+1)\n    f1 = 2*precision*recall/np.maximum(precision+recall,1e-12)\n    accuracy = (tp+(1-y).sum()-fp)/len(y)\n    floor = np.minimum.reduce([precision,recall,f1,accuracy])\n    i = int(np.argmax(floor))\n    return {\'threshold\':float(ps[ends[i]]), \'calibration_min_metric\':float(floor[i]),\n            \'calibration_all_metrics_ge_80\':bool(np.max(floor) >= .80),\n            \'calibration_all_metrics_ge_85\':bool(np.max(floor) >= .85)}\n\n\ndef text_manifest(df):\n    parts = []\n    for fit_year in (2018,2020,2021):\n        pool = training_pool(df,fit_year)\n        parts += [balanced(pool,seed) for seed in SEEDS]\n    for start,end in [(\'2020-01-01\',\'2020-12-31\'),(\'2022-01-01\',\'2022-12-31\'),\n                      (\'2023-01-01\',\'2023-09-30\'),(\'2024-10-01\',\'2025-09-30\')]:\n        parts.append(date_part(df,start,end))\n    return pd.concat(parts).drop_duplicates(\'adsh\')[[\'adsh\',\'cik\',\'form\',\'filed_date\']].sort_values(\'adsh\')\n', 'sec_v5_evidence.py': '"""Public SEC evidence: complete quarterly indexes and optional ORIGINAL text.\n\nOnly used for predictor enrichment. Existing proxy outcome labels are retained.\nNo contact identity is stored. One sequential client, at most two requests/sec.\n"""\nimport gzip\nimport hashlib\nimport io\nimport importlib.metadata\nimport json\nimport re\nimport sqlite3\nimport time\nfrom pathlib import Path\nfrom contextlib import closing\nfrom urllib.error import HTTPError, URLError\nfrom urllib.request import Request, urlopen\nfrom zipfile import ZipFile\nimport numpy as np\nimport pandas as pd\n\nFORMS = {\'10-K\',\'10-Q\',\'10-K/A\',\'10-Q/A\',\'8-K\',\'8-K/A\',\'NT 10-K\',\'NT 10-Q\'}\nRISK_PATTERNS = {\n    \'weakness\':r\'\\bmaterial weaknesses?\\b\',\n    \'ineffective\':r\'\\b(?:not effective|ineffective)\\b\',\n    \'deficiency\':r\'\\bsignificant deficienc(?:y|ies)\\b\',\n    \'going_concern\':r\'\\bgoing concern\\b\',\n    \'restatement\':r\'\\brestat(?:ement|ements|ed)\\b\',\n    \'related_party\':r\'\\brelated.part(?:y|ies)\\b\',\n    \'estimate\':r\'\\b(?:critical accounting|accounting estimates?)\\b\',\n    \'late\':r\'\\b(?:late filing|untimely|unable to timely)\\b\',\n    \'auditor_change\':r\'\\b(?:auditor resignation|dismissed.*?auditor)\\b\',\n    \'remediation\':r\'\\bremediat(?:ion|e|ed)\\b\',\n    \'controls\':r\'\\b(?:internal controls?|disclosure controls?)\\b\',\n}\n\n\nclass SecClient:\n    def __init__(self, user_agent):\n        if not re.search(r\'\\S+@\\S+\\.\\S+\', user_agent) or \' \' not in user_agent.strip():\n            raise ValueError(\'SEC_USER_AGENT must contain a real name/organization and contact email.\')\n        self.user_agent = user_agent.strip()\n        self.previous = 0.0\n\n    def get(self, url, limit=80*1024**2, original_form=None):\n        if not url.startswith(\'https://www.sec.gov/Archives/\'):\n            raise ValueError(\'Only public SEC archive URLs are allowed.\')\n        for attempt in range(4):\n            time.sleep(max(0, .5-(time.monotonic()-self.previous)))\n            self.previous = time.monotonic()\n            try:\n                request = Request(url, headers={\'User-Agent\':self.user_agent,\'Accept-Encoding\':\'identity\'})\n                with urlopen(request, timeout=90) as response:\n                    chunks=[]; size=0\n                    while True:\n                        block=response.read(64*1024)\n                        if not block:\n                            break\n                        chunks.append(block); size+=len(block)\n                        if size > limit:\n                            raise ValueError(\'SEC response exceeded the configured size limit.\')\n                        if original_form and b\'</DOCUMENT>\' in block.upper():\n                            joined=b\'\'.join(chunks)\n                            if original_document(joined,original_form) is not None:\n                                return joined\n                    return b\'\'.join(chunks)\n            except HTTPError as exc:\n                if exc.code == 403:\n                    raise RuntimeError(\'SEC denied access (403). Stop requests; use an allowed connection later. No bypass is attempted.\') from exc\n                if exc.code not in (429,500,502,503,504):\n                    raise\n                if attempt == 3:\n                    raise RuntimeError(\'SEC rate/server error persisted. Cached progress is retained; retry later.\') from exc\n                time.sleep(min(60,10*2**attempt))\n            except (URLError,TimeoutError) as exc:\n                if attempt == 3:\n                    raise RuntimeError(\'Network unavailable; cached progress is retained.\') from exc\n                time.sleep(3*2**attempt)\n\n\ndef parse_master(raw):\n    records=[]\n    for line in raw.decode(\'latin-1\').splitlines():\n        fields=line.split(\'|\')\n        if len(fields) != 5 or not fields[0].isdigit() or fields[2] not in FORMS:\n            continue\n        cik,_,form,date,path=fields\n        match=re.fullmatch(r\'edgar/data/(\\d+)/(\\d{10}-\\d{2}-\\d{6})\\.txt\',path)\n        if not match or int(match[1]) != int(cik):\n            continue\n        records.append((str(int(cik)),form,date,match[2],path))\n    if not records:\n        raise ValueError(\'SEC index did not contain expected filing records.\')\n    frame=pd.DataFrame(records,columns=[\'cik\',\'form\',\'filed_date\',\'adsh\',\'archive_path\'])\n    frame[\'filed_date\']=pd.to_datetime(frame.filed_date,errors=\'raise\')\n    return frame.drop_duplicates([\'cik\',\'adsh\'])\n\n\ndef load_sec_history(cache, client=None):\n    cache=Path(cache); cache.mkdir(parents=True,exist_ok=True)\n    parts=[]\n    # Predictor dates end in 2025Q3. Later index events are unnecessary.\n    for year in range(2009,2026):\n        for quarter in range(1,5):\n            if year==2025 and quarter>3:\n                break\n            path=cache/f\'{year}Q{quarter}.parquet\'\n            if not path.exists():\n                if client is None:\n                    raise FileNotFoundError(f\'Run the evidence notebook first: missing {path.name}\')\n                url=f\'https://www.sec.gov/Archives/edgar/full-index/{year}/QTR{quarter}/master.zip\'\n                blob=client.get(url)\n                with ZipFile(io.BytesIO(blob)) as z:\n                    member=next((n for n in z.namelist() if n.lower()==\'master.idx\'),None)\n                    if member is None:\n                        raise ValueError(\'Unexpected SEC index ZIP layout.\')\n                    part=parse_master(z.read(member))\n                temp=path.with_suffix(\'.part.parquet\')\n                part.to_parquet(temp,index=False); temp.replace(path)\n                print(f\'Cached {year}Q{quarter}: {len(part):,} relevant filings\',flush=True)\n            part=pd.read_parquet(path)\n            parts.append(part)\n    return pd.concat(parts,ignore_index=True).drop_duplicates([\'cik\',\'adsh\'])\n\n\ndef original_document(blob, form):\n    if form not in (\'10-K\',\'10-Q\'):\n        raise ValueError(\'Text predictors must be an original 10-K or 10-Q.\')\n    text=blob.decode(\'utf-8\',errors=\'replace\')\n    for doc in re.findall(r\'<DOCUMENT>(.*?)</DOCUMENT>\',text,flags=re.I|re.S):\n        kind=re.search(r\'<TYPE>\\s*([^\\r\\n<]+)\',doc,flags=re.I)\n        body=re.search(r\'<TEXT>(.*?)</TEXT>\',doc,flags=re.I|re.S)\n        if kind and body and kind[1].strip().upper()==form:\n            return body[1]\n    return None\n\n\ndef compact_text(html):\n    from bs4 import BeautifulSoup\n    soup=BeautifulSoup(html,\'html.parser\')\n    for tag in soup.find_all([\'script\',\'style\',\'ix:header\',\'ix:hidden\']):\n        tag.decompose()\n    plain=re.sub(r\'\\s+\',\' \',soup.get_text(\' \',strip=True))\n    words=len(plain.split())\n    if words<200:\n        return \'\', {\'text_available\':0.0,\'text_word_log\':float(np.log1p(words))}\n    features={\'text_available\':1.0,\'text_word_log\':float(np.log1p(words))}\n    # Compact excerpts are screening windows, not certified SEC item sections.\n    excerpts=[plain[:1000]]\n    for name,pattern in RISK_PATTERNS.items():\n        matches=list(re.finditer(pattern,plain,flags=re.I))\n        features[f\'text_{name}_per_1k\']=len(matches)*1000/max(words,1)\n        for match in matches[:2]:\n            excerpts.append(plain[max(0,match.start()-250):match.end()+500])\n    return \'\\n\'.join(dict.fromkeys(excerpts))[:10000],features\n\n\ndef text_db(path):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    db=sqlite3.connect(path)\n    db.execute(\'CREATE TABLE IF NOT EXISTS texts (adsh TEXT PRIMARY KEY, url TEXT, status TEXT, body BLOB, lexical TEXT, sha256 TEXT)\')\n    db.commit()\n    return db\n\n\ndef collect_text(manifest, db_path, client, max_new=3000):\n    db=text_db(db_path)\n    done={r[0] for r in db.execute(\'SELECT adsh FROM texts\')}\n    pending=manifest[~manifest.adsh.isin(done)]\n    print(f\'Text cache: {len(manifest)-len(pending):,}/{len(manifest):,} attempted; \'\n          f\'{len(pending):,} pending. This session limit: {max_new or "all"}.\',flush=True)\n    try:\n        for i,row in enumerate(pending.itertuples(index=False),1):\n            if max_new and i>max_new:\n                break\n            if not re.fullmatch(r\'\\d{10}-\\d{2}-\\d{6}\',row.adsh):\n                raise ValueError(\'Invalid accession in text manifest.\')\n            url=f\'https://www.sec.gov/Archives/edgar/data/{int(row.cik)}/{row.adsh}.txt\'\n            body=\'\'; features={\'text_available\':0.0}; digest=\'\'\n            try:\n                blob=client.get(url,limit=25*1024**2,original_form=row.form)\n                html=original_document(blob,row.form)\n                if html is None:\n                    status=\'original_document_missing\'\n                else:\n                    body,features=compact_text(html)\n                    digest=hashlib.sha256(html.encode()).hexdigest()\n                    status=\'ok\' if body else \'insufficient_text\'\n            except HTTPError as exc:\n                if exc.code != 404:\n                    raise\n                status=\'http_404\'\n            except ValueError as exc:\n                if \'size limit\' not in str(exc):\n                    raise\n                status=\'size_limit\'\n            db.execute(\'INSERT OR REPLACE INTO texts VALUES (?,?,?,?,?,?)\',\n                       (row.adsh,url,status,gzip.compress(body.encode()),json.dumps(features),digest))\n            if i%25==0:\n                db.commit(); print(f\'Text requests completed this session: {i:,}\',flush=True)\n    finally:\n        db.commit(); db.close()\n    return text_coverage(manifest,db_path)\n\n\ndef text_coverage(manifest,db_path):\n    with closing(text_db(db_path)) as db:\n        states=dict(db.execute(\'SELECT adsh,status FROM texts\'))\n    status=manifest.adsh.map(states)\n    report={\'requested\':len(manifest),\'attempted\':int(status.notna().sum()),\n            \'usable\':int(status.eq(\'ok\').sum()),\'pending\':int(status.isna().sum())}\n    print(\'Text coverage:\',report,flush=True)\n    return report\n\n\ndef gpu_text_features(manifest,db_path,cache):\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA required for text embedding.\')\n    coverage=text_coverage(manifest,db_path)\n    if coverage[\'pending\']:\n        raise RuntimeError(\'Text download is incomplete. Resume the evidence notebook, or explicitly set USE_TEXT=False.\')\n    if coverage[\'usable\']/coverage[\'requested\'] < .90:\n        raise RuntimeError(\'Text coverage below 90%; inspect failures before enabling the text experiment.\')\n    from sentence_transformers import SentenceTransformer\n    from huggingface_hub import model_info\n    cache=Path(cache); cache.mkdir(parents=True,exist_ok=True)\n    meta_path=cache/\'encoder.json\'\n    name=\'sentence-transformers/all-MiniLM-L6-v2\'\n    if meta_path.exists():\n        meta=json.loads(meta_path.read_text())\n    else:\n        meta={\'model\':name,\'revision\':model_info(name).sha,\'license\':\'Apache-2.0\',\n              \'source\':f\'https://huggingface.co/{name}\',\'text_extraction_version\':1}\n        meta_path.write_text(json.dumps(meta,indent=2))\n    encoder=SentenceTransformer(meta[\'model\'],revision=meta[\'revision\'],device=\'cuda\',trust_remote_code=False)\n    frames=[]\n    with closing(text_db(db_path)) as db:\n        for start in range(0,len(manifest),1000):\n            ids=manifest.adsh.iloc[start:start+1000].tolist()\n            raw=[db.execute(\'SELECT body,lexical,sha256 FROM texts WHERE adsh=?\',(a,)).fetchone() for a in ids]\n            packages={n:importlib.metadata.version(n) for n in\n                      (\'torch\',\'sentence-transformers\',\'transformers\')}\n            key=hashlib.sha256(json.dumps([ids,[r[2] for r in raw],meta,packages],sort_keys=True).encode()).hexdigest()[:24]\n            path=cache/f\'{key}.parquet\'\n            if path.exists():\n                frames.append(pd.read_parquet(path)); continue\n            pieces=[]; owners=[]; lex=[]\n            for i,(body,features,_) in enumerate(raw):\n                lex.append(json.loads(features)); words=gzip.decompress(body).decode().split()\n                for offset in range(0,min(len(words),8*140),140):\n                    pieces.append(\' \'.join(words[offset:offset+140])); owners.append(i)\n            vectors=np.zeros((len(ids),384),dtype=np.float32); counts=np.zeros(len(ids))\n            if pieces:\n                encoded=encoder.encode(pieces,batch_size=64,show_progress_bar=True,\n                                       convert_to_numpy=True,normalize_embeddings=True)\n                np.add.at(vectors,np.asarray(owners),encoded)\n                np.add.at(counts,np.asarray(owners),1)\n                vectors/=np.maximum(counts[:,None],1)\n            frame=pd.concat([pd.DataFrame({\'adsh\':ids}),pd.DataFrame(lex),\n                             pd.DataFrame(vectors,columns=[f\'text_emb_{j}\' for j in range(384)])],axis=1)\n            temporary=path.with_suffix(\'.part.parquet\')\n            frame.to_parquet(temporary,index=False); temporary.replace(path)\n            frames.append(frame)\n    del encoder\n    torch.cuda.empty_cache()\n    return pd.concat(frames,ignore_index=True).fillna(0)\n\n\nif __name__==\'__main__\':\n    import argparse\n    import getpass\n    import os\n    import research_v5_core as core\n    parser=argparse.ArgumentParser(description=\'Cache public SEC evidence without fitting any model.\')\n    parser.add_argument(\'--root\',default=\'.\')\n    parser.add_argument(\'--text\',action=\'store_true\',help=\'Also download optional original-filing excerpts.\')\n    parser.add_argument(\'--max-new\',type=int,default=3000,help=\'Text requests this session; 0 means all pending.\')\n    args=parser.parse_args()\n    if args.max_new<0:\n        parser.error(\'--max-new must be nonnegative\')\n    root=Path(args.root)\n    client=SecClient(os.environ.get(\'SEC_USER_AGENT\') or getpass.getpass(\'Real SEC User-Agent (name/organization and email): \'))\n    events=load_sec_history(root/\'evidence_v5/indexes\',client)\n    print(f\'Index cache complete: {len(events):,} relevant events.\')\n    if args.text:\n        path=root/\'data/colab_amendment_proxy.csv.gz\'\n        if not path.exists():\n            path=root/\'colab_amendment_proxy.csv.gz\'\n        manifest=core.text_manifest(core.load_data(path))\n        manifest.to_csv(root/\'evidence_v5/text_manifest.csv.gz\',index=False,compression=\'gzip\')\n        collect_text(manifest,root/\'evidence_v5/original_text.sqlite\',client,args.max_new)\n', '19_mda_advanced.py': '"""Collect full MD&A sections from the public SEC archive for the GPU study.\n\nThis module only prepares data. It never fits a model and never changes labels.\nThe source manifest is the existing, fixed text manifest; all outcomes are\nrejoined from the current schema-v4 dataset by accession number.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport hashlib\nimport json\nimport re\nimport sqlite3\nimport sys\nfrom pathlib import Path\nfrom io import TextIOWrapper\nfrom contextlib import closing\n\nimport pandas as pd\nfrom bs4 import BeautifulSoup\n\nSRC = Path(__file__).resolve().parent\nif str(SRC) not in sys.path:\n    sys.path.insert(0, str(SRC))\n\nimport research_v5_core as core\nimport sec_v5_evidence as evidence\n\nMAX_SECTION_WORDS = 6000\nMIN_SECTION_WORDS = 200\n\n\ndef _item_pattern(form: str, start: bool) -> re.Pattern[str]:\n    if form == \'10-K\':\n        item = r\'7\' if start else r\'(?:7\\s*A|8)\'\n    elif form == \'10-Q\':\n        item = r\'2\' if start else r\'(?:3|4)\'\n    else:\n        raise ValueError(\'Only original 10-K and 10-Q filings have supported MD&A items.\')\n    return re.compile(r\'^\\s*(?:PART\\s+[IVX]+\\s+)?ITEM\\s+\' + item + r\'(?![A-Z0-9])\', re.I)\n\n\ndef extract_mda_section(html: str, form: str, max_words: int = MAX_SECTION_WORDS) -> tuple[str, str, int]:\n    """Extract the longest well-bounded Item 7/Item 2 section.\n\n    SEC filings often repeat section labels in a table of contents. Candidate\n    start/end pairs are scored by body length, which generally selects the real\n    section. Ambiguous or short documents are marked unavailable instead of\n    silently substituting the whole filing.\n    """\n    if form not in {\'10-K\', \'10-Q\'}:\n        raise ValueError(\'Only original 10-K and 10-Q filings are accepted.\')\n    if max_words < MIN_SECTION_WORDS:\n        raise ValueError(f\'max_words must be at least {MIN_SECTION_WORDS}.\')\n    soup = BeautifulSoup(html or \'\', \'html.parser\')\n    for tag in soup.find_all([\'script\', \'style\', \'ix:header\', \'ix:hidden\']):\n        tag.decompose()\n    lines = [re.sub(r\'\\s+\', \' \', line).strip() for line in soup.get_text(\'\\n\').splitlines()]\n    lines = [line for line in lines if line]\n    if not lines:\n        return \'\', \'no_text\', 0\n\n    starts = _item_pattern(form, True)\n    ends = _item_pattern(form, False)\n    start_indices = [i for i, line in enumerate(lines) if starts.search(line)]\n    candidates = []\n    for start in start_indices:\n        end = next((i for i in range(start + 1, len(lines)) if ends.search(lines[i])), None)\n        if end is None:\n            continue\n        body = \'\\n\'.join(lines[start + 1:end]).strip()\n        words = body.split()\n        if len(words) >= MIN_SECTION_WORDS:\n            candidates.append((len(words), body))\n    if not candidates:\n        return \'\', \'no_mda_section\', 0\n\n    word_count, body = max(candidates, key=lambda item: item[0])\n    if word_count > max_words:\n        body = \' \'.join(body.split()[:max_words])\n        return body, \'ok_truncated\', max_words\n    return body, \'ok\', word_count\n\n\ndef build_manifest(root: Path) -> pd.DataFrame:\n    """Return the fixed 111k-style study manifest with current proxy labels."""\n    root = Path(root)\n    proxy_path = root / \'data\' / \'colab_amendment_proxy.csv.gz\'\n    if not proxy_path.is_file():\n        raise FileNotFoundError(f\'Missing prepared proxy dataset: {proxy_path}\')\n    cache_dir = root / \'evidence_v5\' / \'mda_sections\'\n    manifest_path = cache_dir / \'text_manifest.csv.gz\'\n    metadata_path = cache_dir / \'text_manifest.json\'\n    dataset_hash = core.file_hash(proxy_path)\n    if manifest_path.is_file() and metadata_path.is_file():\n        metadata = json.loads(metadata_path.read_text(encoding=\'utf-8\'))\n        if metadata.get(\'proxy_sha256\') == dataset_hash:\n            manifest = pd.read_csv(manifest_path, compression=\'gzip\',\n                                   dtype={\'adsh\':str,\'cik\':str,\'form\':str,\'y_proxy\':\'int8\'})\n            manifest[\'filed_date\'] = pd.to_datetime(manifest.filed_date, errors=\'raise\')\n            return manifest\n    proxy = core.load_data(proxy_path)\n    manifest = core.text_manifest(proxy)\n    labels = proxy[[\'adsh\', \'cik\', \'form\', \'filed_date\', core.TARGET]].copy()\n    manifest = manifest[[\'adsh\']].merge(labels, on=\'adsh\', how=\'left\', validate=\'one_to_one\')\n    if manifest[core.TARGET].isna().any() or manifest.cik.isna().any():\n        raise ValueError(\'A selected accession has no current proxy label or issuer ID.\')\n    manifest = manifest.rename(columns={core.TARGET: \'y_proxy\'})\n    manifest[\'cik\'] = manifest.cik.astype(str).str.lstrip(\'0\').replace(\'\', \'0\')\n    manifest[\'filed_date\'] = pd.to_datetime(manifest.filed_date, errors=\'raise\')\n    if manifest.adsh.duplicated().any():\n        raise ValueError(\'The text manifest must contain unique accessions.\')\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    temporary = manifest_path.with_suffix(\'.part.csv.gz\')\n    manifest.to_csv(temporary, index=False, compression={\'method\':\'gzip\',\'compresslevel\':6,\'mtime\':0})\n    temporary.replace(manifest_path)\n    metadata_path.write_text(json.dumps({\'proxy_sha256\':dataset_hash,\'rows\':len(manifest)},indent=2),\n                             encoding=\'utf-8\')\n    return manifest\n\n\ndef open_section_db(path: Path) -> sqlite3.Connection:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    db = sqlite3.connect(path)\n    db.execute(\'\'\'CREATE TABLE IF NOT EXISTS mda_sections (\n        adsh TEXT PRIMARY KEY, cik TEXT NOT NULL, form TEXT NOT NULL,\n        filed_date TEXT NOT NULL, status TEXT NOT NULL, body BLOB NOT NULL,\n        word_count INTEGER NOT NULL, sha256 TEXT NOT NULL)\'\'\')\n    db.commit()\n    return db\n\n\ndef collect_sections(root: Path, user_agent: str, max_new: int = 3000) -> dict:\n    """Fetch/resume only selected original filings, respecting the SEC client."""\n    if max_new < 0:\n        raise ValueError(\'max_new must be zero (all remaining) or a positive integer.\')\n    root = Path(root)\n    manifest = build_manifest(root)\n    db_path = root / \'evidence_v5\' / \'mda_sections\' / \'sections.sqlite\'\n    db = open_section_db(db_path)\n    done = {row[0] for row in db.execute(\'SELECT adsh FROM mda_sections\')}\n    pending = manifest.loc[~manifest.adsh.isin(done)]\n    client = evidence.SecClient(user_agent)\n    attempted = 0\n    try:\n        for row in pending.itertuples(index=False):\n            if max_new and attempted >= max_new:\n                break\n            attempted += 1\n            adsh = str(row.adsh)\n            cik = str(int(row.cik))\n            accession_path = adsh.replace(\'-\', \'\')\n            url = f\'https://www.sec.gov/Archives/edgar/data/{cik}/{accession_path}/{adsh}.txt\'\n            status = \'http_404\'\n            body = \'\'\n            word_count = 0\n            digest = \'\'\n            try:\n                submission = client.get(url, limit=25 * 1024**2, original_form=row.form)\n                document = evidence.original_document(submission, row.form)\n                if document is None:\n                    status = \'original_document_missing\'\n                else:\n                    body, status, word_count = extract_mda_section(document, row.form)\n                    if body:\n                        digest = hashlib.sha256(body.encode(\'utf-8\')).hexdigest()\n            except Exception as exc:\n                from urllib.error import HTTPError\n                if isinstance(exc, HTTPError) and exc.code == 404:\n                    status = \'http_404\'\n                elif isinstance(exc, ValueError) and \'size limit\' in str(exc):\n                    status = \'size_limit\'\n                else:\n                    raise\n            db.execute(\'INSERT INTO mda_sections VALUES (?,?,?,?,?,?,?,?)\',\n                       (adsh, cik, row.form, pd.Timestamp(row.filed_date).strftime(\'%Y-%m-%d\'),\n                        status, sqlite3.Binary(gzip.compress(body.encode(\'utf-8\'))), word_count, digest))\n            if attempted % 25 == 0:\n                db.commit()\n                print(f\'SEC filing requests this session: {attempted:,}/{len(pending):,}\', flush=True)\n        db.commit()\n    finally:\n        db.commit()\n        db.close()\n    return coverage(root, manifest=manifest)\n\n\ndef coverage(root: Path, manifest: pd.DataFrame | None = None) -> dict:\n    manifest = manifest if manifest is not None else build_manifest(root)\n    db_path = Path(root) / \'evidence_v5\' / \'mda_sections\' / \'sections.sqlite\'\n    if not db_path.is_file():\n        return {\'requested\': len(manifest), \'attempted\': 0, \'usable\': 0, \'pending\': len(manifest)}\n    with closing(open_section_db(db_path)) as db:\n        statuses = dict(db.execute(\'SELECT adsh,status FROM mda_sections\'))\n    status = manifest.adsh.map(statuses)\n    return {\'requested\': len(manifest), \'attempted\': int(status.notna().sum()),\n            \'usable\': int(status.isin([\'ok\', \'ok_truncated\']).sum()),\n            \'pending\': int(status.isna().sum()),\n            \'status_counts\': status.fillna(\'pending\').value_counts().sort_index().to_dict()}\n\n\ndef export_dataset(root: Path, output_path: Path | None = None) -> Path:\n    """Export all manifest rows, including explicit missing-text statuses."""\n    root = Path(root)\n    manifest = build_manifest(root)\n    db_path = root / \'evidence_v5\' / \'mda_sections\' / \'sections.sqlite\'\n    if not db_path.is_file():\n        raise FileNotFoundError(f\'No SEC text cache exists yet: {db_path}\')\n    with closing(open_section_db(db_path)) as db:\n        statuses = dict(db.execute(\'SELECT adsh,status FROM mda_sections\'))\n        missing = int((~manifest.adsh.isin(statuses)).sum())\n        if missing:\n            raise RuntimeError(f\'{missing:,} filings remain pending; resume collection before export.\')\n        rows = db.execute(\'SELECT adsh,status,body,word_count,sha256 FROM mda_sections\')\n        cached = {adsh: (status, body, words, digest)\n                  for adsh, status, body, words, digest in rows}\n    output_path = output_path or root / \'data\' / \'colab_mda_text_expanded.csv.gz\'\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    with output_path.open(\'wb\') as raw:\n        with gzip.GzipFile(fileobj=raw, mode=\'wb\', compresslevel=6, mtime=0) as compressed:\n            with TextIOWrapper(compressed, encoding=\'utf-8\', newline=\'\') as text_file:\n                writer = csv.writer(text_file)\n                writer.writerow([\'adsh\', \'cik\', \'form\', \'filed_date\', \'y_proxy\', \'mda_text\',\n                                 \'text_status\', \'mda_word_count\', \'text_sha256\'])\n                for row in manifest.itertuples(index=False):\n                    status, body, words, digest = cached[row.adsh]\n                    section = gzip.decompress(body).decode(\'utf-8\') if body else \'\'\n                    writer.writerow([row.adsh, row.cik, row.form,\n                                     pd.Timestamp(row.filed_date).strftime(\'%Y-%m-%d\'),\n                                     int(row.y_proxy), section, status, int(words), digest])\n    return output_path\n\n\ndef prepare(root: Path, user_agent: str, max_new: int = 3000) -> tuple[dict, Path | None]:\n    report = collect_sections(root, user_agent, max_new=max_new)\n    output = None\n    if report[\'pending\'] == 0:\n        output = export_dataset(root)\n        report[\'dataset\'] = str(output)\n        report[\'dataset_bytes\'] = output.stat().st_size\n    return report, output\n\n', '11_validate_restatement_labels.py': '"""Validate the human-reviewed confirmed-restatement ledger.\n\nPending rows are permitted in the working queue but are never model labels.\nThis validator checks fields and source provenance, not whether a reviewer\'s\naccounting judgment is correct.\n"""\nfrom pathlib import Path\nimport argparse\nimport pandas as pd\n\n\nREQUIRED = {\n    "original_adsh", "original_filed_date", "label", "review_status", "event_type",\n    "event_date", "outcome_followup_through", "evidence_url", "evidence_excerpt",\n    "reviewer", "review_notes",\n}\nALLOWED_EVENT_TYPES = {\n    "restatement", "revision_not_restatement", "other_amendment",\n    "no_confirmed_restatement",\n}\n\n\ndef validate_ledger(frame: pd.DataFrame, min_followup_years: int = 0) -> list[str]:\n    """Return structural errors; pending rows are allowed but excluded from fitting."""\n    errors = []\n    if min_followup_years < 0:\n        return ["min_followup_years cannot be negative"]\n    missing = REQUIRED - set(frame.columns)\n    if missing:\n        return [f"Missing required columns: {sorted(missing)}"]\n    ids = frame.original_adsh.fillna("").astype(str).str.strip()\n    if ids.eq("").any():\n        errors.append("original_adsh must be present on every queue row")\n    if ids[ids.ne("")].duplicated().any():\n        errors.append("original_adsh must be unique (one final adjudication per filing)")\n\n    status = frame.review_status.fillna("").astype(str).str.strip().str.lower()\n    if (~status.isin(["pending", "reviewed"])).any():\n        errors.append("review_status must be pending or reviewed")\n    reviewed = frame.loc[status.eq("reviewed")].copy()\n    if reviewed.empty:\n        return errors\n\n    labels = pd.to_numeric(reviewed.label, errors="coerce")\n    if labels.isna().any() or not labels.isin([0, 1]).all():\n        errors.append("Every reviewed row needs label 0 or 1; pending/unknown rows stay pending")\n    event_types = reviewed.event_type.fillna("").astype(str).str.strip()\n    if not event_types.isin(ALLOWED_EVENT_TYPES).all():\n        errors.append(f"Reviewed event_type must be one of {sorted(ALLOWED_EVENT_TYPES)}")\n    expected_label = event_types.eq("restatement").astype(int)\n    if labels.notna().all() and not labels.astype(int).eq(expected_label).all():\n        errors.append("label must be 1 only for event_type=restatement and 0 otherwise")\n\n    original_dates = pd.to_datetime(reviewed.original_filed_date, errors="coerce")\n    event_dates = pd.to_datetime(reviewed.event_date, errors="coerce")\n    followup = pd.to_datetime(reviewed.outcome_followup_through, errors="coerce")\n    if original_dates.isna().any():\n        errors.append("Reviewed rows need a parseable original_filed_date")\n    if followup.isna().any():\n        errors.append("Every reviewed row needs outcome_followup_through")\n    positives = labels.eq(1)\n    event_required = event_types.ne("no_confirmed_restatement")\n    if (event_required & event_dates.isna()).any():\n        errors.append("Reviewed amendment/restatement events need event_date")\n    if (positives & event_dates.isna()).any():\n        errors.append("Positive restatement rows need event_date")\n    if (positives & event_dates.lt(original_dates)).any():\n        errors.append("A restatement event cannot precede its original filing")\n    if (event_dates.notna() & event_dates.gt(followup)).any():\n        errors.append("outcome_followup_through cannot precede the documented event_date")\n    if (followup.notna() & original_dates.notna() & followup.lt(original_dates)).any():\n        errors.append("outcome_followup_through cannot precede the original filing")\n\n    for column in ("evidence_url", "evidence_excerpt", "reviewer"):\n        values = reviewed[column].fillna("").astype(str).str.strip()\n        if values.eq("").any():\n            errors.append(f"Every reviewed row needs {column}")\n    urls = reviewed.evidence_url.fillna("").astype(str).str.lower()\n    if (~urls.str.startswith("https://www.sec.gov/")).any():\n        errors.append("Evidence URLs must point to a primary SEC.gov filing source")\n\n    if min_followup_years:\n        required_through = original_dates + pd.DateOffset(years=min_followup_years)\n        # If a confirmed event occurred after the requested horizon, its event\n        # date establishes a negative at that horizon, provided it is observed.\n        event_after_horizon = event_dates.notna() & event_dates.gt(required_through)\n        event_within_horizon = event_types.eq("restatement") & event_dates.le(required_through)\n        too_early = followup.lt(required_through) & ~event_after_horizon & ~event_within_horizon\n        if too_early.any():\n            errors.append(f"Reviewed negatives/early events need at least {min_followup_years} years of outcome follow-up")\n    return errors\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("ledger", type=Path, help="Human-reviewed outcome ledger CSV")\n    parser.add_argument("--horizon-years", type=int, default=0,\n                        help="Optionally require mature follow-up for this horizon (1, 3, or 5)")\n    args = parser.parse_args()\n    if not args.ledger.is_file():\n        raise FileNotFoundError(args.ledger)\n    frame = pd.read_csv(args.ledger, dtype={"original_adsh": str})\n    errors = validate_ledger(frame, args.horizon_years)\n    if errors:\n        for error in errors:\n            print(f"ERROR: {error}")\n        raise SystemExit(1)\n    status = frame.review_status.fillna("").astype(str).str.lower()\n    reviewed = frame.loc[status.eq("reviewed")]\n    labels = pd.to_numeric(reviewed.label, errors="coerce").dropna().astype(int)\n    print(f"Valid queue: {len(frame):,} rows; reviewed={len(reviewed):,}; pending={(~status.eq(\'reviewed\')).sum():,}.")\n    if len(labels):\n        print(f"Reviewed outcomes: positives={labels.sum():,}; negatives={(labels == 0).sum():,}; reviewed prevalence={labels.mean():.3%}")\n    print("This checks structure and source links only. A human remains responsible for label decisions and complete follow-up review.")\n\n\nif __name__ == "__main__":\n    main()\n', '22_restatement_study.py': '"""Prepare and evaluate a confirmed-restatement study from reviewed SEC evidence.\n\nNo outcomes are inferred from amendments. Only rows explicitly marked reviewed\nin the manual ledger enter a cohort. Event dates derive 1/3/5-year targets;\nnegative outcomes require documented follow-up through the selected horizon.\nText downloads are limited to reviewed original filings and their prior\nsame-issuer/same-form filings.\n"""\nfrom __future__ import annotations\n\nfrom contextlib import closing\nfrom pathlib import Path\nimport gzip\nimport hashlib\nimport importlib.metadata\nimport importlib.util\nimport json\nimport re\nimport sqlite3\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nSRC = Path(__file__).resolve().parent\nROOT = SRC.parent\nsys.path.insert(0, str(SRC)) if str(SRC) not in sys.path else None\n\nEVENT_TYPES = {\n    "restatement", "revision_not_restatement", "other_amendment",\n    "no_confirmed_restatement",\n}\nHORIZONS = (1, 3, 5)\nTRAJECTORY_RATIOS = (\n    "liabilities_to_assets", "current_assets_to_current_liabilities",\n    "working_capital_to_assets", "cash_to_assets", "receivables_to_revenue",\n    "inventory_to_revenue", "gross_margin", "operating_margin", "net_margin",\n    "net_income_to_assets", "operating_cashflow_to_assets",\n    "operating_cashflow_to_net_income", "accruals_to_assets", "debt_to_assets",\n    "liabilities_to_equity", "retained_earnings_to_assets",\n    "capex_to_operating_cashflow", "revenue_to_assets",\n)\nEMBEDDING_CHUNK_WORDS = 140\nLEDGER_COLUMNS = {\n    "original_adsh", "original_filed_date", "label", "review_status", "event_type",\n    "event_date", "outcome_followup_through", "evidence_url", "evidence_excerpt",\n    "reviewer", "review_notes",\n}\nNON_FEATURES = {\n    "adsh", "cik", "period", "period_date", "filed", "filed_date", "filed_year",\n    "sic", "afs", "wksi", "days_to_amendment", "y_amendment_proxy",\n    "feature_schema_version", "label_observation_cutoff", "label", "event_type",\n    "event_date", "outcome_followup_through", "review_status", "evidence_url",\n    "evidence_excerpt", "reviewer", "review_notes", "sampling_group",\n}\n\n\ndef validate_ledger(frame: pd.DataFrame) -> list[str]:\n    """Validate the working queue and its reviewed rows; pending rows are allowed."""\n    errors = []\n    missing = LEDGER_COLUMNS - set(frame.columns)\n    if missing:\n        return [f"Missing required ledger columns: {sorted(missing)}"]\n    ids = frame.original_adsh.fillna("").astype(str).str.strip()\n    if ids.eq("").any() or ids[ids.ne("")].duplicated().any():\n        errors.append("Each queue row needs a unique original_adsh")\n    statuses = frame.review_status.fillna("").astype(str).str.strip().str.lower()\n    if (~statuses.isin(["pending", "reviewed"])).any():\n        errors.append("review_status must be pending or reviewed")\n    reviewed = frame.loc[statuses.eq("reviewed")].copy()\n    if reviewed.empty:\n        return errors\n    labels = pd.to_numeric(reviewed.label, errors="coerce")\n    if labels.isna().any() or not labels.isin([0, 1]).all():\n        errors.append("Every reviewed row needs label 0 or 1")\n    event_type = reviewed.event_type.fillna("").astype(str).str.strip()\n    if not event_type.isin(EVENT_TYPES).all():\n        errors.append(f"Reviewed event_type must be one of {sorted(EVENT_TYPES)}")\n    if labels.notna().all() and not labels.astype(int).eq(event_type.eq("restatement").astype(int)).all():\n        errors.append("label must be 1 exactly when event_type=restatement")\n    original = pd.to_datetime(reviewed.original_filed_date, errors="coerce")\n    event = pd.to_datetime(reviewed.event_date, errors="coerce")\n    followup = pd.to_datetime(reviewed.outcome_followup_through, errors="coerce")\n    if original.isna().any() or followup.isna().any():\n        errors.append("Reviewed rows need parseable original_filed_date and outcome_followup_through")\n    event_required = event_type.ne("no_confirmed_restatement")\n    if (event_required & event.isna()).any():\n        errors.append("A documented amendment/restatement event needs event_date")\n    if ((event.notna()) & event.lt(original)).any():\n        errors.append("event_date cannot precede original_filed_date")\n    if ((event.notna()) & event.gt(followup)).any():\n        errors.append("outcome_followup_through cannot precede event_date")\n    if ((followup.notna()) & followup.lt(original)).any():\n        errors.append("outcome_followup_through cannot precede original_filed_date")\n    for col in ("evidence_url", "evidence_excerpt", "reviewer"):\n        if reviewed[col].fillna("").astype(str).str.strip().eq("").any():\n            errors.append(f"Every reviewed row needs {col}")\n    urls = reviewed.evidence_url.fillna("").astype(str).str.lower()\n    if (~urls.str.startswith("https://www.sec.gov/")).any():\n        errors.append("evidence_url must be a primary SEC.gov filing URL")\n    return errors\n\n\ndef derive_horizon_labels(ledger: pd.DataFrame, horizon_years: int) -> tuple[pd.DataFrame, int]:\n    """Return reviewed rows observable at H years, with a horizon-specific target."""\n    if horizon_years not in HORIZONS:\n        raise ValueError(f"horizon_years must be one of {HORIZONS}")\n    errors = validate_ledger(ledger)\n    if errors:\n        raise ValueError("Invalid review ledger: " + "; ".join(errors))\n    reviewed = ledger.loc[ledger.review_status.fillna("").astype(str).str.lower().eq("reviewed")].copy()\n    if reviewed.empty:\n        return reviewed.assign(target_restatement=pd.Series(dtype="int8")), 0\n    original = pd.to_datetime(reviewed.original_filed_date, errors="raise")\n    end = original + pd.DateOffset(years=horizon_years)\n    event = pd.to_datetime(reviewed.event_date, errors="coerce")\n    followup = pd.to_datetime(reviewed.outcome_followup_through, errors="coerce")\n    is_restatement = reviewed.event_type.astype(str).eq("restatement")\n    positive = is_restatement & event.le(end)\n    # A later documented restatement is a valid horizon-negative only if the\n    # review establishes it occurred after H; other negatives need full H follow-up.\n    event_later = event.notna() & event.gt(end)\n    mature = positive | (event_later & followup.ge(event)) | followup.ge(end)\n    ready = reviewed.loc[mature].copy()\n    ready["target_restatement"] = positive.loc[mature].astype("int8").to_numpy()\n    ready["horizon_end"] = end.loc[mature].dt.strftime("%Y-%m-%d").to_numpy()\n    ready["outcome_followup_through"] = followup.loc[mature].dt.strftime("%Y-%m-%d").to_numpy()\n    return ready.reset_index(drop=True), int((~mature).sum())\n\n\ndef validate_proxy_features(proxy: pd.DataFrame) -> None:\n    """Require the reviewed study\'s known point-in-time feature schema."""\n    required={"adsh","cik","form","filed_date","feature_schema_version","label_observation_cutoff"}\n    missing=required-set(proxy.columns)\n    if missing:\n        raise ValueError(f"Prepared feature table is missing required schema-4 columns: {sorted(missing)}")\n    if not pd.to_numeric(proxy.feature_schema_version,errors="coerce").eq(4).all():\n        raise ValueError("Use the prepared feature_schema_version=4 table")\n    if not proxy.adsh.is_unique or proxy.adsh.isna().any():\n        raise ValueError("Prepared feature table needs one unique accession per row")\n    if not set(proxy.form.dropna().astype(str)) <= {"10-K","10-Q"}:\n        raise ValueError("Predictors must come from original 10-K/10-Q filings")\n    cutoff=pd.to_datetime(proxy.label_observation_cutoff,errors="coerce")\n    if cutoff.isna().any() or cutoff.nunique()!=1:\n        raise ValueError("Prepared feature table needs one common parseable label observation cutoff")\n\n\ndef _load_module(name: str, path: Path):\n    spec = importlib.util.spec_from_file_location(name, path)\n    module = importlib.util.module_from_spec(spec)\n    assert spec.loader is not None\n    spec.loader.exec_module(module)\n    return module\n\n\ndef _prior_filing_links(proxy: pd.DataFrame, targets: pd.DataFrame) -> pd.DataFrame:\n    """Select the immediately prior same-issuer/form filing, strictly by date."""\n    rows = []\n    history = proxy[["adsh", "cik", "form", "filed_date"]].copy()\n    history["cik_norm"] = history.cik.astype(str).str.lstrip("0").replace("", "0")\n    history["filed_date"] = pd.to_datetime(history.filed_date, errors="raise")\n    history = history.sort_values(["cik_norm", "form", "filed_date", "adsh"])\n    buckets = {(str(cik), str(form)): group for (cik, form), group in\n               history.groupby(["cik_norm", "form"], sort=False)}\n    for row in targets.itertuples(index=False):\n        cik = str(row.cik).lstrip("0") or "0"\n        group = buckets.get((cik, str(row.original_form)))\n        prior = ""\n        if group is not None:\n            earlier = group.loc[group.filed_date.lt(pd.Timestamp(row.original_filed_date))]\n            if not earlier.empty:\n                prior = str(earlier.iloc[-1].adsh)\n        rows.append((str(row.original_adsh), prior))\n    return pd.DataFrame(rows, columns=["original_adsh", "prior_adsh"])\n\n\ndef _financial_trajectory_features(proxy: pd.DataFrame, targets: pd.DataFrame,\n                                    max_prior_filings: int = 3) -> pd.DataFrame:\n    """Build point-in-time trends from the current and up to three earlier same-form filings.\n\n    Only a fixed set of financial ratios is read. For each target filing, every\n    historical observation must have a strictly earlier filing date. This adds\n    issuer-specific financial movement without using the future outcome or\n    any amendment/review metadata.\n    """\n    if max_prior_filings < 1:\n        raise ValueError("max_prior_filings must be positive")\n    ratio_cols = [name for name in TRAJECTORY_RATIOS if name in proxy.columns]\n    if not ratio_cols:\n        return pd.DataFrame({"adsh": targets.adsh.astype(str)})\n    required = {"adsh", "cik", "form", "filed_date"}\n    if not required.issubset(proxy.columns) or not required.issubset(targets.columns):\n        raise ValueError("Financial history needs adsh, cik, form, and filed_date")\n\n    history = proxy[["adsh", "cik", "form", "filed_date", *ratio_cols]].copy()\n    history["adsh"] = history.adsh.astype(str)\n    history["cik_norm"] = history.cik.astype(str).str.lstrip("0").replace("", "0")\n    history["filed_date"] = pd.to_datetime(history.filed_date, errors="coerce")\n    if history.filed_date.isna().any() or history.adsh.duplicated().any():\n        raise ValueError("Financial history needs unique accessions and parseable filing dates")\n    for name in ratio_cols:\n        history[name] = pd.to_numeric(history[name], errors="coerce").replace(\n            [np.inf, -np.inf], np.nan)\n    history = history.sort_values(["cik_norm", "form", "filed_date", "adsh"])\n    groups = {(str(cik), str(form)): group for (cik, form), group in\n              history.groupby(["cik_norm", "form"], sort=False)}\n\n    result = []\n    for target in targets.itertuples(index=False):\n        cik = str(target.cik).lstrip("0") or "0"\n        filed = pd.Timestamp(target.filed_date)\n        group = groups.get((cik, str(target.form)))\n        earlier = (group.loc[group.filed_date.lt(filed)].tail(max_prior_filings)\n                   if group is not None else history.iloc[:0])\n        current = {name: pd.to_numeric(getattr(target, name, np.nan), errors="coerce")\n                   for name in ratio_cols}\n        record = {"adsh": str(target.adsh),\n                  "trajectory_prior_filing_count": int(len(earlier))}\n        if len(earlier):\n            record["trajectory_days_since_prior_filing"] = int(\n                (filed - earlier.filed_date.iloc[-1]).days)\n        else:\n            record["trajectory_days_since_prior_filing"] = np.nan\n        for name in ratio_cols:\n            prior_values = earlier[name].to_numpy(dtype=float) if len(earlier) else np.array([])\n            current_value = current[name]\n            values = np.r_[prior_values, current_value]\n            valid = np.isfinite(values)\n            x = np.arange(len(values), dtype=float)[valid]\n            y = values[valid]\n            prefix = f"trajectory__{name}__"\n            record[prefix + "slope_per_filing"] = (\n                float(np.polyfit(x, y, 1)[0]) if len(y) >= 2 else np.nan)\n            record[prefix + "std"] = float(np.std(y)) if len(y) >= 2 else np.nan\n            record[prefix + "range"] = float(np.ptp(y)) if len(y) >= 2 else np.nan\n            finite_prior = prior_values[np.isfinite(prior_values)]\n            record[prefix + "latest_step"] = (\n                float(current_value - finite_prior[-1])\n                if np.isfinite(current_value) and len(finite_prior) else np.nan)\n        result.append(record)\n    return pd.DataFrame(result)\n\n\ndef section_db(path: Path) -> sqlite3.Connection:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    db = sqlite3.connect(path)\n    db.execute("""CREATE TABLE IF NOT EXISTS restatement_mda (\n        adsh TEXT PRIMARY KEY, cik TEXT NOT NULL, form TEXT NOT NULL,\n        filed_date TEXT NOT NULL, status TEXT NOT NULL, body BLOB NOT NULL,\n        word_count INTEGER NOT NULL, sha256 TEXT NOT NULL)""")\n    db.commit()\n    return db\n\n\ndef collect_reviewed_mda(proxy: pd.DataFrame, ledger: pd.DataFrame, horizon_years: int,\n                         db_path: Path, user_agent: str, max_new: int = 500,\n                         client=None) -> dict:\n    """Fetch only reviewed target originals and their prior comparable originals."""\n    ready, immature = derive_horizon_labels(ledger, horizon_years)\n    if ready.empty:\n        raise ValueError("No reviewed rows have an observable outcome at this horizon.")\n    proxy = proxy.copy()\n    validate_proxy_features(proxy)\n    proxy["filed_date"] = pd.to_datetime(proxy.filed_date, errors="raise")\n    proxy["cik"] = proxy.cik.astype(str).str.lstrip("0").replace("", "0")\n    if not proxy.adsh.is_unique:\n        raise ValueError("Prepared feature table must have unique accession IDs")\n    target_ids = set(ready.original_adsh.astype(str))\n    targets = proxy.loc[proxy.adsh.astype(str).isin(target_ids),\n                        ["adsh", "cik", "form", "filed_date"]].rename(columns={\n                            "adsh":"original_adsh", "form":"original_form",\n                            "filed_date":"proxy_filed_date"})\n    if len(targets) != len(ready):\n        missing = target_ids - set(targets.original_adsh.astype(str))\n        raise ValueError(f"{len(missing)} reviewed accessions do not match the prepared feature table; first={sorted(missing)[:3]}")\n    targets = ready[["original_adsh", "original_filed_date"]].merge(\n        targets, on="original_adsh", validate="one_to_one")\n    targets["original_filed_date"] = pd.to_datetime(targets.original_filed_date, errors="raise")\n    targets["proxy_filed_date"] = pd.to_datetime(targets.proxy_filed_date, errors="raise")\n    if not targets.original_filed_date.eq(targets.proxy_filed_date).all():\n        raise ValueError("The reviewed original_filed_date does not match the prepared accession record")\n    if not targets.original_form.isin(["10-K", "10-Q"]).all():\n        raise ValueError("Only original 10-K and 10-Q filings may supply predictors")\n    links = _prior_filing_links(proxy, targets)\n    target_manifest = targets.rename(columns={"original_adsh":"adsh"})[\n        ["adsh", "cik", "original_form", "original_filed_date"]].rename(columns={\n            "original_form":"form", "original_filed_date":"filed_date"})\n    prior_ids = set(links.prior_adsh) - {""}\n    prior_manifest = proxy.loc[proxy.adsh.astype(str).isin(prior_ids),\n        ["adsh", "cik", "form", "filed_date"]].copy()\n    manifest = pd.concat([target_manifest, prior_manifest], ignore_index=True).drop_duplicates("adsh")\n    manifest["adsh"] = manifest.adsh.astype(str)\n    manifest["cik"] = manifest.cik.astype(str).str.lstrip("0").replace("", "0")\n    manifest["filed_date"] = pd.to_datetime(manifest.filed_date, errors="raise")\n\n    evidence = _load_module("restatement_sec_evidence", SRC / "sec_v5_evidence.py")\n    advanced = _load_module("restatement_mda_extractor", SRC / "19_mda_advanced.py")\n    client = client or evidence.SecClient(user_agent)\n    db = section_db(db_path)\n    done = {r[0] for r in db.execute("SELECT adsh FROM restatement_mda")}\n    pending = manifest.loc[~manifest.adsh.isin(done)]\n    attempted = 0\n    try:\n        for row in pending.itertuples(index=False):\n            if max_new and attempted >= max_new:\n                break\n            attempted += 1\n            cik = str(int(row.cik))\n            accession_path = str(row.adsh).replace("-", "")\n            url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession_path}/{row.adsh}.txt"\n            status, body, words, digest = "", "", 0, ""\n            try:\n                blob = client.get(url, limit=80 * 1024**2, original_form=row.form)\n                html = evidence.original_document(blob, row.form)\n                if html is None:\n                    status = "original_document_missing"\n                else:\n                    body, status, words = advanced.extract_mda_section(html, row.form)\n                    digest = hashlib.sha256(html.encode("utf-8", errors="replace")).hexdigest()\n            except Exception as exc:\n                # Access denial, rate limits, and network errors remain visible\n                # and stop the run; missing/oversize documents are cached below.\n                from urllib.error import HTTPError\n                if isinstance(exc, HTTPError) and exc.code == 404:\n                    status = "http_404"\n                elif isinstance(exc, ValueError) and "size limit" in str(exc):\n                    status = "size_limit"\n                else:\n                    raise\n            db.execute("INSERT OR REPLACE INTO restatement_mda VALUES (?,?,?,?,?,?,?,?)",\n                       (row.adsh, cik, row.form, pd.Timestamp(row.filed_date).strftime("%Y-%m-%d"),\n                        status, gzip.compress(body.encode("utf-8")), int(words), digest))\n            if attempted % 25 == 0:\n                db.commit()\n                print(f"SEC filing texts cached: {attempted:,}/{len(pending):,} in this session", flush=True)\n    finally:\n        db.commit()\n        db.close()\n    with closing(section_db(db_path)) as db:\n        states = dict(db.execute("SELECT adsh,status FROM restatement_mda"))\n    counts = pd.Series([states.get(a, "pending") for a in manifest.adsh]).value_counts().to_dict()\n    return {"horizon_years":horizon_years, "reviewed_mature_rows":len(ready),\n            "excluded_immature_rows":immature, "target_rows":len(targets),\n            "prior_filings":len(prior_ids), "unique_filings":len(manifest),\n            "attempted_this_session":attempted, "status_counts":counts,\n            "pending":int(sum(v for k,v in counts.items() if k == "pending"))}\n\n\ndef prepare_model_data(proxy_path: Path, ledger_path: Path, sections_db_path: Path,\n                       horizon_years: int, output_path: Path) -> dict:\n    """Join reviewed horizon labels, point-in-time financials, and cached MD&A."""\n    ledger = pd.read_csv(ledger_path, dtype={"original_adsh":str}, low_memory=False)\n    ready, immature = derive_horizon_labels(ledger, horizon_years)\n    if ready.empty:\n        raise ValueError("No reviewed outcomes have sufficient follow-up; review labels first.")\n    proxy = pd.read_csv(proxy_path, dtype={"adsh":str,"cik":str}, low_memory=False)\n    validate_proxy_features(proxy)\n    if not proxy.adsh.is_unique:\n        raise ValueError("Prepared feature table contains duplicate accessions")\n    ids = set(ready.original_adsh.astype(str))\n    part = proxy.loc[proxy.adsh.astype(str).isin(ids)].copy()\n    if len(part) != len(ready):\n        raise ValueError("Some reviewed accessions are absent from the prepared feature table")\n    part = part.merge(ready[["original_adsh", "target_restatement", "horizon_end",\n                             "outcome_followup_through"]].rename(columns={"original_adsh":"adsh"}),\n                      on="adsh", validate="one_to_one")\n    trajectory = _financial_trajectory_features(proxy, part)\n    part = part.merge(trajectory, on="adsh", validate="one_to_one")\n    part["original_adsh"] = part.adsh.astype(str)\n    part["filed_date"] = pd.to_datetime(part.filed_date, errors="raise")\n    link_proxy = proxy[["adsh", "cik", "form", "filed_date"]].copy()\n    link_proxy["cik"] = link_proxy.cik.astype(str).str.lstrip("0").replace("", "0")\n    targets = part[["adsh", "cik", "form", "filed_date"]].rename(columns={\n        "adsh":"original_adsh", "form":"original_form", "filed_date":"original_filed_date"})\n    targets["cik"] = targets.cik.astype(str).str.lstrip("0").replace("", "0")\n    links = _prior_filing_links(link_proxy, targets)\n    part = part.merge(links, on="original_adsh", validate="one_to_one")\n    with closing(section_db(sections_db_path)) as db:\n        texts = {a:(gzip.decompress(body).decode("utf-8", errors="replace"), status)\n                 for a,body,status in db.execute("SELECT adsh,body,status FROM restatement_mda")}\n    part["mda_text"] = part.adsh.astype(str).map(lambda a:texts.get(a,("","missing"))[0])\n    part["mda_status"] = part.adsh.astype(str).map(lambda a:texts.get(a,("","pending"))[1])\n    part["prior_mda_text"] = part.prior_adsh.astype(str).map(lambda a:texts.get(a,("","missing"))[0])\n    part["prior_mda_status"] = part.prior_adsh.astype(str).map(lambda a:texts.get(a,("","pending"))[1])\n    # Never pass label, future outcome, accession, issuer ID, dates, or the\n    # existing amendment proxy target to predictors. Past issuer histories and\n    # financial deltas remain legitimate as-filed features.\n    keep_meta = {"adsh", "cik", "form", "fp", "sic2", "filed_date", "target_restatement",\n                 "horizon_end", "outcome_followup_through", "prior_adsh", "mda_text",\n                 "mda_status", "prior_mda_text", "prior_mda_status"}\n    feature_cols = [c for c in part if c not in NON_FEATURES and c not in keep_meta and\n                    not c.startswith(("label_", "target_", "evidence_", "event_"))]\n    meta_order = [c for c in ["adsh", "cik", "form", "fp", "sic2", "filed_date",\n        "target_restatement", "horizon_end", "outcome_followup_through", "prior_adsh",\n        "mda_text", "mda_status", "prior_mda_text", "prior_mda_status"] if c in part]\n    safe = part[meta_order + feature_cols].copy()\n    if safe.adsh.duplicated().any():\n        raise ValueError("Prepared model table must have one row per original filing")\n    safe["filed_year"] = pd.to_datetime(safe.filed_date).dt.year.astype("int16")\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    temp = output_path.with_name(output_path.name + ".part")\n    safe.to_csv(temp, index=False, compression="gzip")\n    temp.replace(output_path)\n    return {"output":str(output_path), "rows":len(safe),\n            "positives":int(safe.target_restatement.sum()),\n            "negatives":int((safe.target_restatement == 0).sum()),\n            "excluded_immature":immature,\n            "current_mda_usable":int(safe.mda_status.isin(["ok", "ok_truncated"]).sum()),\n            "prior_mda_usable":int(safe.prior_mda_status.isin(["ok", "ok_truncated"]).sum()),\n            "feature_count":len(feature_cols)}\n\n\ndef _lexical_features(text: str) -> dict[str, float]:\n    text = (text or "").lower()\n    words = max(1, len(re.findall(r"\\b[\\w\'-]+\\b", text)))\n    cues = {\n        "restatement":r"\\brestat(?:e|ed|ement|ements|ing)\\w*\\b",\n        "error":r"\\b(?:error|errors|erroneous|misstatement|misstatements)\\b",\n        "nonreliance":r"\\bnon.?reliance\\b|should not be relied upon",\n        "material_weakness":r"\\bmaterial weaknesses?\\b",\n        "control_deficiency":r"\\b(?:significant deficiency|ineffective internal control|control deficiencies)\\b",\n        "accounting_estimate":r"\\b(?:accounting estimate|critical accounting|judgment|assumption)\\w*\\b",\n        "uncertainty":r"\\b(?:uncertain|uncertainty|may materially|could materially|reasonably possible)\\b",\n        "investigation":r"\\b(?:investigation|audit committee|subpoena|inquiry)\\w*\\b",\n        "remediation":r"\\bremediat\\w*\\b",\n        "revision":r"\\b(?:revision|revised|correction|corrected|adjustment)\\w*\\b",\n        "forward_looking":r"\\b(?:expect|anticipate|believe|forecast|project|estimate)\\w*\\b",\n    }\n    out = {f"lex_{name}_per_1k":len(re.findall(pattern,text))/words*1000\n           for name,pattern in cues.items()}\n    out["lex_word_count_log"] = float(np.log1p(words - 1))\n    return out\n\n\ndef _document_chunks(text: str, chunk_words: int = EMBEDDING_CHUNK_WORDS) -> list[str]:\n    """Split all available MD&A text into non-overlapping encoder-sized chunks."""\n    if chunk_words < 1:\n        raise ValueError("chunk_words must be positive")\n    words = (text or "").split()\n    return [" ".join(words[start:start + chunk_words])\n            for start in range(0, len(words), chunk_words)]\n\n\ndef _embed_sections(frame: pd.DataFrame):\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA is required. In Colab select Runtime > Change runtime type > T4 GPU.")\n    from sentence_transformers import SentenceTransformer\n    from huggingface_hub import model_info\n    encoder_name="sentence-transformers/all-MiniLM-L6-v2"\n    encoder_revision=model_info(encoder_name).sha\n    model = SentenceTransformer(encoder_name,revision=encoder_revision,device="cuda",\n                                trust_remote_code=False)\n    texts = pd.concat([frame.mda_text.fillna(""), frame.prior_mda_text.fillna("")],\n                      ignore_index=True).astype(str).tolist()\n    vecs = np.zeros((len(texts), 384), dtype=np.float32)\n    chunks, owners = [], []\n    for idx, text in enumerate(texts):\n        # The filing extractor retains up to 6,000 words. Encode the entire\n        # available section instead of silently discarding everything after\n        # its first 1,120 words.\n        for chunk in _document_chunks(text):\n            chunks.append(chunk)\n            owners.append(idx)\n    if chunks:\n        encoded = model.encode(chunks, batch_size=64, show_progress_bar=True,\n                               convert_to_numpy=True, normalize_embeddings=True)\n        counts = np.zeros(len(texts), dtype=np.float32)\n        for owner, vector in zip(owners, encoded):\n            vecs[owner] += vector\n            counts[owner] += 1\n        nonzero = counts > 0\n        vecs[nonzero] /= counts[nonzero,None]\n        norms = np.linalg.norm(vecs,axis=1)\n        vecs[nonzero] /= np.maximum(norms[nonzero,None],1e-8)\n    current, prior = vecs[:len(frame)], vecs[len(frame):]\n    current_lex = pd.DataFrame([_lexical_features(t) for t in frame.mda_text.fillna("")])\n    prior_lex = pd.DataFrame([_lexical_features(t) for t in frame.prior_mda_text.fillna("")])\n    lexical_names = list(current_lex.columns)\n    current_lex_values = current_lex.to_numpy(dtype=np.float32)\n    prior_lex_values = prior_lex.to_numpy(dtype=np.float32)\n    lexical_delta = current_lex_values - prior_lex_values\n    delta = current - prior\n    cosine = np.sum(current*prior,axis=1).reshape(-1,1)\n    novelty = np.concatenate([delta, np.abs(delta), cosine],axis=1).astype(np.float32)\n    status = frame.mda_status.isin(["ok","ok_truncated"]).to_numpy(dtype=np.float32).reshape(-1,1)\n    prior_status = frame.prior_mda_status.isin(["ok","ok_truncated"]).to_numpy(dtype=np.float32).reshape(-1,1)\n    current_embedding_names = [f"mda_current_embedding__{i:03d}" for i in range(current.shape[1])]\n    prior_embedding_names = [f"mda_prior_embedding__{i:03d}" for i in range(prior.shape[1])]\n    change_embedding_names = [f"mda_change_embedding__{i:03d}" for i in range(delta.shape[1])]\n    abs_change_names = [f"mda_abs_change_embedding__{i:03d}" for i in range(delta.shape[1])]\n    current_lex_names = [f"lex_current__{name}" for name in lexical_names]\n    prior_lex_names = [f"lex_prior__{name}" for name in lexical_names]\n    delta_lex_names = [f"lex_change__{name}" for name in lexical_names]\n    availability_names = ["availability__current_mda", "availability__prior_mda"]\n    text_names = (current_embedding_names + current_lex_names + availability_names[:1])\n    novelty_names = (current_embedding_names + prior_embedding_names + change_embedding_names +\n                     abs_change_names + ["mda_embedding_cosine_similarity"] + current_lex_names +\n                     prior_lex_names + availability_names)\n    interpretable_names = current_lex_names + prior_lex_names + delta_lex_names + availability_names\n    text_view = np.concatenate([current,current_lex_values,status],axis=1)\n    full_text = np.concatenate([current,prior,novelty,current_lex_values,\n                                prior_lex_values],axis=1)\n    interpretable = np.concatenate([current_lex_values,prior_lex_values,lexical_delta,\n                                    status,prior_status],axis=1)\n    return ({"text":text_view,"text_names":text_names,\n             "novelty":np.concatenate([full_text,status,prior_status],axis=1),\n             "novelty_names":novelty_names,\n             "interpretable":interpretable,"interpretable_names":interpretable_names},\n            {"name":encoder_name,"revision":encoder_revision,"license":"Apache-2.0",\n             "source_section_max_words":6000,"embedding_chunk_words":EMBEDDING_CHUNK_WORDS,\n             "embedding_coverage":"all chunks across the available MD&A section",\n             "lexical_features":lexical_names})\n\n\ndef _archive_topic_features(frame: pd.DataFrame, archive_path: Path,\n                             holdout_start_year: int, n_components: int = 48,\n                             min_df: int = 5, max_features: int = 30000):\n    """Learn an unsupervised MD&A vocabulary/topics from pre-holdout corpus text.\n\n    The archive\'s legacy proxy labels are deliberately never read. Only corpus\n    documents filed before the temporal test window fit TF-IDF and SVD. The\n    learned transform is then applied to reviewed current/prior MD&A sections.\n    """\n    from sklearn.decomposition import TruncatedSVD\n    from sklearn.feature_extraction.text import TfidfVectorizer\n\n    archive_path=Path(archive_path)\n    required={"adsh","filed_date","mda_text"}\n    archive=pd.read_csv(archive_path,compression="infer",usecols=sorted(required),\n                        dtype={"adsh":str},low_memory=False)\n    if archive.adsh.isna().any() or archive.adsh.astype(str).duplicated().any():\n        raise ValueError("Full MD&A archive needs one unique, nonblank accession per row")\n    archive["filed_date"]=pd.to_datetime(archive.filed_date,errors="coerce")\n    if archive.filed_date.isna().any():\n        raise ValueError("Full MD&A archive contains unparseable filed_date values")\n    archive["mda_text"]=archive.mda_text.fillna("").astype(str)\n    fit=archive.loc[archive.filed_date.dt.year.lt(int(holdout_start_year))].copy()\n    fit=fit.loc[fit.mda_text.str.split().str.len().ge(200)]\n    if len(fit)<max(20,min_df*2):\n        raise ValueError(f"Only {len(fit)} usable pre-holdout archive documents; need a broader text archive")\n    vectorizer=TfidfVectorizer(\n        lowercase=True,strip_accents="unicode",sublinear_tf=True,\n        ngram_range=(1,2),min_df=min_df,max_df=.995,max_features=max_features,\n        dtype=np.float32,token_pattern=r"(?u)\\b[\\w][\\w\'-]*\\b")\n    matrix=vectorizer.fit_transform(fit.mda_text.tolist())\n    components=min(int(n_components),matrix.shape[0]-1,matrix.shape[1]-1)\n    if components<2:\n        raise ValueError("Pre-holdout archive text has too few distinct terms for topic features")\n    svd=TruncatedSVD(n_components=components,n_iter=5,random_state=20261021)\n    svd.fit(matrix)\n\n    current=frame.mda_text.fillna("").astype(str).tolist()\n    prior=frame.prior_mda_text.fillna("").astype(str).tolist()\n    projected=svd.transform(vectorizer.transform(current+prior)).astype(np.float32)\n    current_topics=projected[:len(frame)]\n    prior_topics=projected[len(frame):]\n    delta=current_topics-prior_topics\n    cosine=np.sum(current_topics*prior_topics,axis=1,keepdims=True)\n    topics=np.concatenate([current_topics,prior_topics,delta,np.abs(delta),cosine],axis=1)\n    names=([f"archive_current_topic__{i:03d}" for i in range(components)] +\n           [f"archive_prior_topic__{i:03d}" for i in range(components)] +\n           [f"archive_change_topic__{i:03d}" for i in range(components)] +\n           [f"archive_abs_change_topic__{i:03d}" for i in range(components)] +\n           ["archive_topic_cosine_similarity"])\n    matched_ids=(set(frame.adsh.astype(str)) if "adsh" in frame else set())\n    meta={"archive_sha256":_file_sha256(archive_path),"archive_rows":int(len(archive)),\n          "archive_rows_matching_reviewed_originals":int(archive.adsh.astype(str).isin(matched_ids).sum()),\n          "archive_fit_rows":int(len(fit)),"archive_rows_excluded_as_holdout_or_later":\n              int(archive.filed_date.dt.year.ge(int(holdout_start_year)).sum()),\n          "fit_through_year":int(holdout_start_year)-1,"holdout_start_year":int(holdout_start_year),\n          "vocabulary_size":int(len(vectorizer.vocabulary_)),"topic_components":int(components),\n          "legacy_proxy_labels_read":False,\n          "method":"word unigram/bigram TF-IDF plus truncated SVD, fit on pre-holdout MD&A only"}\n    return {"archive_topics":topics,"archive_topic_names":names},meta,vectorizer,svd\n\n\ndef _features(train: pd.DataFrame, valid: pd.DataFrame, view: str,\n              embeddings: dict[str,np.ndarray], train_idx: np.ndarray, valid_idx: np.ndarray,\n              return_names: bool = False):\n    from sklearn.preprocessing import OneHotEncoder\n    numeric = [c for c in train.columns if c not in NON_FEATURES and not c.startswith((\n        "target_", "label_", "event_", "evidence_", "candidate_", "review_", "sampling_")) and c not in {\n        "adsh","cik","form","fp","sic2","filed_date","filed_year","target_restatement",\n        "horizon_end","outcome_followup_through","prior_adsh","mda_text","mda_status",\n        "prior_mda_text","prior_mda_status","_embed_pos"} and pd.api.types.is_numeric_dtype(train[c])]\n    categorical = [c for c in ("form","fp","sic2") if c in train]\n    feature_names = ([f"structured__{c}" for c in numeric] +\n                     [f"structured_missing__{c}" for c in numeric])\n    def numeric_matrix(frame):\n        vals = frame[numeric].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan)\n        missing = vals.isna().to_numpy(dtype=np.float32)\n        medians = train[numeric].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan).median().fillna(0)\n        # Fit imputation values only on the training fold.\n        vals = vals.fillna(medians).clip(-1e12,1e12).to_numpy(dtype=np.float32)\n        return np.concatenate([vals,missing],axis=1)\n    numeric_train, numeric_valid = numeric_matrix(train), numeric_matrix(valid)\n    if categorical:\n        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)\n        cat_train=encoder.fit_transform(train[categorical].fillna("unknown").astype(str))\n        cat_valid=encoder.transform(valid[categorical].fillna("unknown").astype(str))\n        numeric_train=np.concatenate([numeric_train,cat_train],axis=1)\n        numeric_valid=np.concatenate([numeric_valid,cat_valid],axis=1)\n        feature_names += [f"categorical__{name}" for name in encoder.get_feature_names_out(categorical)]\n    if view == "financial":\n        result=(numeric_train,numeric_valid,feature_names)\n    elif view == "mda":\n        result=(embeddings["text"][train_idx],embeddings["text"][valid_idx],\n                embeddings["text_names"])\n    elif view == "interpretable":\n        result=(np.concatenate([numeric_train,embeddings["interpretable"][train_idx]],axis=1),\n                np.concatenate([numeric_valid,embeddings["interpretable"][valid_idx]],axis=1),\n                feature_names+embeddings["interpretable_names"])\n    elif view == "full_archive":\n        result=(np.concatenate([numeric_train,embeddings["novelty"][train_idx],\n                                embeddings["archive_topics"][train_idx]],axis=1),\n                np.concatenate([numeric_valid,embeddings["novelty"][valid_idx],\n                                embeddings["archive_topics"][valid_idx]],axis=1),\n                feature_names+embeddings["novelty_names"]+embeddings["archive_topic_names"])\n    else:\n        result=(np.concatenate([numeric_train,embeddings["novelty"][train_idx]],axis=1),\n                np.concatenate([numeric_valid,embeddings["novelty"][valid_idx]],axis=1),\n                feature_names+embeddings["novelty_names"])\n    return result if return_names else result[:2]\n\n\ndef _balance(frame: pd.DataFrame, seed: int) -> pd.DataFrame:\n    """Make a transparent 1:1 year/form matched set, with no replacement."""\n    parts=[]\n    keys=[c for c in ("filed_year","form") if c in frame]\n    if not keys:\n        keys=["form"] if "form" in frame else []\n    groups=[((),frame)] if not keys else frame.groupby(keys,dropna=False,sort=True)\n    for _,group in groups:\n        pos=group[group.target_restatement.eq(1)]\n        neg=group[group.target_restatement.eq(0)]\n        n=min(len(pos),len(neg))\n        if n:\n            parts.extend([pos.sample(n,random_state=seed),neg.sample(n,random_state=seed+1)])\n    if not parts:\n        return frame.iloc[:0].copy()\n    return pd.concat(parts).sample(frac=1,random_state=seed+2).reset_index(drop=True)\n\n\ndef _class_weights(frame: pd.DataFrame) -> np.ndarray:\n    """Use all training rows while giving each outcome class equal total weight."""\n    y=frame.target_restatement.to_numpy(dtype=np.int8)\n    counts=np.bincount(y,minlength=2)\n    if np.any(counts==0):\n        raise ValueError("Both outcome classes are required to compute training weights")\n    weights=np.where(y==1,len(y)/(2*counts[1]),len(y)/(2*counts[0]))\n    return weights.astype(np.float32)\n\n\ndef temporal_holdout_mask(frame: pd.DataFrame, fraction: float = .20) -> tuple[np.ndarray, int]:\n    """Hold the latest filing years out of fitting and all model selection."""\n    if not 0 < fraction < .5:\n        raise ValueError("fraction must be between zero and one half")\n    years=np.sort(pd.to_datetime(frame.filed_date,errors="raise").dt.year.unique())\n    if len(years)<5:\n        raise ValueError("At least five filing years are required for a future-period holdout")\n    n_hold=max(1,int(np.ceil(len(years)*fraction)))\n    start_year=int(years[-n_hold])\n    return pd.to_datetime(frame.filed_date).dt.year.to_numpy()>=start_year,start_year\n\n\ndef _report(y, p, threshold):\n    from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score\n    pred=(np.asarray(p)>=threshold).astype(int)\n    y=np.asarray(y).astype(int)\n    return {"n":int(len(y)),"positives":int(y.sum()),"negatives":int((y==0).sum()),\n            "accuracy":float(accuracy_score(y,pred)),\n            "precision":float(precision_score(y,pred,zero_division=0)),\n            "recall":float(recall_score(y,pred,zero_division=0)),\n            "f1":float(f1_score(y,pred,zero_division=0)),\n            "roc_auc":float(roc_auc_score(y,p)) if len(np.unique(y))==2 else None,\n            "pr_auc":float(average_precision_score(y,p)) if len(np.unique(y))==2 else None,\n            "threshold":float(threshold)}\n\n\ndef _file_sha256(path: Path) -> str:\n    digest=hashlib.sha256()\n    with open(path,"rb") as stream:\n        for block in iter(lambda:stream.read(1024*1024),b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _assert_cuda_booster(model) -> None:\n    """Fail if a fitted XGBoost member did not retain its CUDA device setting."""\n    config=json.loads(model.get_booster().save_config())\n    devices=[]\n    def visit(node):\n        if isinstance(node,dict):\n            for key,value in node.items():\n                if key=="device": devices.append(str(value))\n                visit(value)\n        elif isinstance(node,list):\n            for value in node: visit(value)\n    visit(config)\n    if not any(value.startswith("cuda") for value in devices):\n        raise RuntimeError(f"XGBoost did not confirm CUDA after fitting; device settings={devices}")\n\n\ndef _assert_catboost_gpu(model) -> None:\n    params=model.get_all_params()\n    if str(params.get("task_type","")).upper()!="GPU":\n        raise RuntimeError("CatBoost did not confirm task_type=GPU after fitting")\n\n\ndef _best_threshold(y, p):\n    candidates=np.unique(np.r_[0.01,np.asarray(p),0.99])\n    scored=[]\n    for threshold in candidates:\n        r=_report(y,p,threshold)\n        minimum=min(r[k] for k in ("accuracy","precision","recall","f1"))\n        scored.append((minimum,-abs(float(threshold)-.5),float(threshold)))\n    return max(scored)[2]\n\n\ndef _bootstrap_metric_intervals(y, p, threshold, seed=20261020, replicates=1000):\n    """Stratified percentile intervals for metrics on the stated test sample."""\n    y=np.asarray(y).astype(int); p=np.asarray(p,dtype=float)\n    pos=np.flatnonzero(y==1); neg=np.flatnonzero(y==0)\n    if not len(pos) or not len(neg):\n        return {}\n    rng=np.random.default_rng(seed)\n    values={name:[] for name in ("accuracy","precision","recall","f1","roc_auc","pr_auc")}\n    for _ in range(replicates):\n        idx=np.r_[rng.choice(pos,len(pos),replace=True),rng.choice(neg,len(neg),replace=True)]\n        report=_report(y[idx],p[idx],threshold)\n        for name in values:\n            if report[name] is not None:\n                values[name].append(report[name])\n    return {name:{"lower_95":float(np.quantile(scores,.025)),\n                  "upper_95":float(np.quantile(scores,.975))}\n            for name,scores in values.items() if scores}\n\n\ndef _paired_auc_difference(y, comparison, baseline, seed=20261020, replicates=1000):\n    """Paired stratified bootstrap for the incremental value of a feature view."""\n    from sklearn.metrics import roc_auc_score, average_precision_score\n    y=np.asarray(y).astype(int)\n    comparison=np.asarray(comparison,dtype=float)\n    baseline=np.asarray(baseline,dtype=float)\n    pos=np.flatnonzero(y==1); neg=np.flatnonzero(y==0)\n    if not len(pos) or not len(neg):\n        return {}\n    rng=np.random.default_rng(seed)\n    point={"roc_auc_difference":float(roc_auc_score(y,comparison)-roc_auc_score(y,baseline)),\n           "pr_auc_difference":float(average_precision_score(y,comparison)-average_precision_score(y,baseline))}\n    samples={name:[] for name in point}\n    for _ in range(replicates):\n        idx=np.r_[rng.choice(pos,len(pos),replace=True),rng.choice(neg,len(neg),replace=True)]\n        samples["roc_auc_difference"].append(\n            float(roc_auc_score(y[idx],comparison[idx])-roc_auc_score(y[idx],baseline[idx])))\n        samples["pr_auc_difference"].append(\n            float(average_precision_score(y[idx],comparison[idx])-\n                  average_precision_score(y[idx],baseline[idx])))\n    return {name:{"estimate":point[name],"lower_95":float(np.quantile(values,.025)),\n                  "upper_95":float(np.quantile(values,.975))}\n            for name,values in samples.items()}\n\n\ndef _save_shap_importance(model, engine, features, labels, feature_names, output_dir):\n    """Write local SHAP rankings and feature-family summaries for the holdout."""\n    if engine=="xgboost":\n        import xgboost as xgb\n        contributions=model.get_booster().predict(xgb.DMatrix(features),pred_contribs=True)[:,:-1]\n    else:\n        from catboost import Pool\n        contributions=model.get_feature_importance(Pool(features,label=np.asarray(labels).astype(int)),\n                                                   type="ShapValues")[:,:-1]\n    if contributions.shape[1] != len(feature_names):\n        raise ValueError("SHAP output does not align with the feature-name schema")\n    importance=pd.DataFrame({"feature":feature_names,\n                             "mean_abs_shap":np.mean(np.abs(contributions),axis=0)})\n    importance["feature_group"]=importance.feature.str.split("__",n=1).str[0]\n    importance.sort_values("mean_abs_shap",ascending=False).to_csv(\n        Path(output_dir)/"feature_importance_shap.csv",index=False)\n    grouped=(importance.groupby("feature_group",as_index=False)\n             .agg(feature_count=("feature","size"),\n                  total_mean_abs_shap=("mean_abs_shap","sum"),\n                  mean_abs_shap_per_feature=("mean_abs_shap","mean"))\n             .sort_values("total_mean_abs_shap",ascending=False))\n    grouped.to_csv(Path(output_dir)/"feature_group_importance_shap.csv",index=False)\n\n\ndef train_gpu(dataset_path: Path, output_dir: Path, seed: int = 20261020,\n              archive_path: Path | None = None, target_column: str = "target_restatement",\n              target_definition: str | None = None) -> dict:\n    """Compare GPU models for a named target with an optional full-archive text view.\n\n    Internally, the established training implementation uses ``target_restatement``\n    as a compatibility alias. The requested target column is retained in outputs,\n    named in the report, and all other target/provenance fields are excluded from\n    predictors.\n    """\n    import torch\n    import xgboost as xgb\n    from catboost import CatBoostClassifier\n    from sklearn.model_selection import StratifiedGroupKFold\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA is required. Select an NVIDIA GPU runtime; CPU fallback is disabled.")\n    if not xgb.build_info().get("USE_CUDA",False):\n        raise RuntimeError("Installed XGBoost was built without CUDA; reinstall the Colab GPU dependencies.")\n    output_dir=Path(output_dir)\n    output_dir.mkdir(parents=True,exist_ok=True)\n    frame=pd.read_csv(dataset_path,compression="gzip",dtype={"adsh":str,"cik":str},low_memory=False)\n    if target_column not in frame:\n        raise ValueError(f"Target column {target_column!r} is missing from the prepared data.")\n    frame=frame.loc[pd.to_numeric(frame[target_column],errors="coerce").notna()].copy()\n    target_values=pd.to_numeric(frame[target_column],errors="raise")\n    if not target_values.isin([0,1]).all():\n        raise ValueError(f"Target column {target_column!r} must contain only binary values.")\n    if target_column != "target_restatement":\n        frame["target_restatement"]=target_values.astype("int8")\n    else:\n        frame["target_restatement"]=target_values.astype("int8")\n    target_definition = target_definition or target_column\n    frame["cik_norm"]=frame.cik.astype(str).str.lstrip("0").replace("","0")\n    frame["filed_year"]=pd.to_datetime(frame.filed_date).dt.year\n    frame["_embed_pos"]=np.arange(len(frame),dtype=np.int64)\n    classes=frame.target_restatement.value_counts()\n    if len(classes)<2 or int(classes.min())<10:\n        raise ValueError(f"Need at least 10 rows per class to begin; counts={classes.to_dict()}.")\n    groups_per_class=frame.groupby("target_restatement").cik_norm.nunique()\n    n_splits=min(5,int(groups_per_class.min()))\n    if n_splits<3:\n        raise ValueError(f"Need at least 3 distinct issuers per class; issuer counts={groups_per_class.to_dict()}.")\n    # Future filing years are excluded from model and threshold selection.\n    held_mask,holdout_start_year=temporal_holdout_mask(frame,.20)\n    embeds,encoder_meta=_embed_sections(frame)\n    archive_meta=None\n    if archive_path is not None:\n        archive_embeddings,archive_meta,archive_vectorizer,archive_svd=_archive_topic_features(\n            frame,archive_path,holdout_start_year)\n        embeds.update(archive_embeddings)\n        import joblib\n        joblib.dump(archive_vectorizer,output_dir/"archive_tfidf_vectorizer.joblib",compress=3)\n        joblib.dump(archive_svd,output_dir/"archive_topic_svd.joblib",compress=3)\n    dev=frame.loc[~held_mask].reset_index(drop=True)\n    hold=frame.loc[held_mask].reset_index(drop=True)\n    if dev.target_restatement.nunique()<2:\n        raise ValueError("Earlier filing years contain only one outcome class; add reviewed labels across more years.")\n    dev_groups=dev.cik_norm.to_numpy()\n    dev_groups_per_class=dev.groupby("target_restatement").cik_norm.nunique()\n    dev_splits=min(n_splits,int(dev_groups_per_class.min()))\n    if dev_splits<3:\n        raise ValueError(f"Earlier years need at least 3 distinct issuers per class for grouped CV; counts={dev_groups_per_class.to_dict()}.")\n    splitter=StratifiedGroupKFold(n_splits=dev_splits,shuffle=True,random_state=seed)\n    # The optional full_archive view adds unsupervised topics learned from the\n    # available pre-holdout corpus; labels remain limited to reviewed outcomes.\n    views=("financial","mda","interpretable","fusion")\n    if archive_path is not None:\n        views += ("full_archive",)\n    configs=[("xgboost",view,3,240) for view in views]\n    configs += [("catboost",view,4,300) for view in views]\n    cv_results=[]\n    config_failures=[]\n    oof={}\n    for model_id,(engine,view,depth,trees) in enumerate(configs):\n        probs=np.full(len(dev),np.nan,dtype=float)\n        failed=False\n        thresholds=[]\n        for fold,(tr_idx,va_idx) in enumerate(splitter.split(dev,dev.target_restatement,dev_groups)):\n            # Retain every development row; class weights balance the loss.\n            tr=dev.iloc[tr_idx].reset_index(drop=True)\n            va=_balance(dev.iloc[va_idx],seed+fold*20+1)\n            if tr.empty or va.empty or va.target_restatement.nunique()<2:\n                continue\n            # Keep original frame row indexes through accession lookup after balanced sampling.\n            train_lookup=dev.set_index("adsh")["_embed_pos"]\n            tr_pos=train_lookup.loc[tr.adsh.astype(str)].to_numpy(dtype=int)\n            va_pos=train_lookup.loc[va.adsh.astype(str)].to_numpy(dtype=int)\n            xtr,xva=_features(tr,va,view,embeds,tr_pos,va_pos)\n            train_weights=_class_weights(tr)\n            if engine=="xgboost":\n                model=xgb.XGBClassifier(n_estimators=trees,max_depth=depth,learning_rate=.035,\n                    min_child_weight=4,subsample=.85,colsample_bytree=.85,reg_lambda=8,\n                    objective="binary:logistic",eval_metric="logloss",tree_method="hist",\n                    device="cuda",random_state=seed+fold,n_jobs=2)\n                model.fit(xtr,tr.target_restatement.astype(int),sample_weight=train_weights)\n                _assert_cuda_booster(model)\n            else:\n                try:\n                    model=CatBoostClassifier(iterations=trees,depth=depth,learning_rate=.035,\n                        loss_function="Logloss",eval_metric="Logloss",task_type="GPU",devices="0",\n                        l2_leaf_reg=8,random_seed=seed+fold,verbose=False,allow_writing_files=False)\n                    model.fit(xtr,tr.target_restatement.astype(int),sample_weight=train_weights,\n                              verbose=False)\n                    _assert_catboost_gpu(model)\n                except Exception as exc:\n                    config_failures.append({"engine":engine,"view":view,"depth":depth,\n                                            "trees":trees,"fold":fold,\n                                            "error":f"{type(exc).__name__}: {exc}"[:500]})\n                    failed=True\n                    break\n            p=model.predict_proba(xva)[:,1]\n            # Map probabilities to the sampled validation subset; use distinct\n            # IDs so no issuer contributes train/test rows in this fold.\n            for adsh,prob in zip(va.adsh.astype(str),p):\n                loc=dev.index[dev.adsh.astype(str).eq(adsh)]\n                if len(loc): probs[loc[0]]=prob\n        if failed:\n            continue\n        usable=np.isfinite(probs)\n        if usable.sum()<20 or dev.loc[usable,"target_restatement"].nunique()<2:\n            continue\n        balanced_oof=_balance(dev.loc[usable].assign(_p=probs[usable]),seed+model_id)\n        ids=balanced_oof.adsh.astype(str)\n        # Reconstruct aligned predictions from the OOF accession map.\n        p_lookup=dict(zip(dev.loc[usable].adsh.astype(str),probs[usable]))\n        balanced_prob=np.asarray([p_lookup[a] for a in ids])\n        threshold=_best_threshold(balanced_oof.target_restatement.to_numpy(),balanced_prob)\n        report=_report(balanced_oof.target_restatement.to_numpy(),balanced_prob,threshold)\n        report.update({"engine":engine,"view":view,"max_depth":depth,"n_estimators":trees,\n                       "oof_rows":int(usable.sum()),"selection_metric_min":min(report[k] for k in ("accuracy","precision","recall","f1"))})\n        cv_results.append(report)\n        oof[(engine,view,depth,trees)]=(probs,threshold)\n    if not cv_results:\n        raise ValueError("No valid issuer-grouped CV folds. Add reviewed outcomes from more issuers/years.")\n    # Select on issuer-grouped development PR-AUC; the forward holdout is untouched.\n    leaderboard=pd.DataFrame(cv_results).sort_values(["pr_auc","roc_auc"],ascending=False)\n    best=leaderboard.iloc[0]\n    best_key=(str(best.engine),str(best.view),int(best.max_depth),int(best.n_estimators))\n    threshold=float(oof[best_key][1])\n    best_by_config={}\n    for engine in ("xgboost","catboost"):\n        for view in views:\n            candidates=leaderboard.loc[leaderboard.view.eq(view)&leaderboard.engine.eq(engine)]\n            if candidates.empty:\n                continue\n            view_best=candidates.iloc[0]\n            key=(str(view_best.engine),str(view_best.view),int(view_best.max_depth),\n                 int(view_best.n_estimators))\n            best_by_config[f"{engine}/{view}"]={"row":view_best,"key":key,\n                                                 "threshold":float(oof[key][1])}\n    hold_bal=_balance(hold,seed+999)\n    result={"gpu":torch.cuda.get_device_name(0),"xgboost":xgb.__version__,"rows":len(frame),\n            "target_column":target_column,"target_definition":target_definition,\n            "horizon_years":(int(match.group(1)) if (match:=re.search(r"h([135])y",Path(dataset_path).name)) else None),\n            "positives":int(frame.target_restatement.sum()),"negatives":int((frame.target_restatement==0).sum()),\n            "current_mda_coverage":float(frame.mda_status.isin(["ok","ok_truncated"]).mean()),\n            "prior_mda_coverage":float(frame.prior_mda_status.isin(["ok","ok_truncated"]).mean()),\n            "issuer_groups":int(frame.cik_norm.nunique()),"holdout_rows":len(hold),\n            "future_holdout_start_year":holdout_start_year,\n            "dataset_sha256":_file_sha256(Path(dataset_path)),\n            "study_code_sha256":_file_sha256(Path(__file__)),\n            "embedding_encoder":encoder_meta,\n            "financial_trajectory_features":{\n                "feature_count":int(sum(str(c).startswith("trajectory__") for c in frame.columns)),\n                "definition":"Current and up to three strictly earlier same-issuer/same-form filings; ratio slopes, dispersion, range, and latest step.",\n                "future_or_same_day_history_used":False},\n            "full_archive_text_features":archive_meta,\n            "library_versions":{name:importlib.metadata.version(dist) for name,dist in\n                (("torch","torch"),("sentence_transformers","sentence-transformers"),\n                 ("scikit_learn","scikit-learn"),("xgboost","xgboost-cu12"),\n                 ("catboost","catboost"))},\n            "holdout_positive_issuers":int(hold.loc[hold.target_restatement.eq(1),"cik_norm"].nunique()),\n            "holdout_negative_issuers":int(hold.loc[hold.target_restatement.eq(0),"cik_norm"].nunique()),\n            "selected_model":{"engine":best_key[0],"view":best_key[1],"max_depth":best_key[2],"n_estimators":best_key[3],"threshold":threshold},\n            "selected_models_by_view":{name:{"engine":item["key"][0],"view":item["key"][1],\n                "max_depth":item["key"][2],"n_estimators":item["key"][3],\n                "threshold":item["threshold"]} for name,item in best_by_config.items()},\n            "development_cv_1_to_1":{k:v for k,v in best.to_dict().items()},\n            "model_selection_metric":"issuer_grouped_development_pr_auc",\n            "gpu_model_failures":config_failures,\n            "holdout_results_by_view":{},\n            "balanced_holdout_1_to_1":None,"curated_holdout_all_rows":None,\n            "holdout_adequate_for_claim":bool(len(hold_bal)>=60 and hold_bal.target_restatement.sum()>=30 and\n                    hold_bal.loc[hold_bal.target_restatement.eq(1),"cik_norm"].nunique()>=10 and\n                    hold_bal.loc[hold_bal.target_restatement.eq(0),"cik_norm"].nunique()>=10),\n            "metric_goal_all_four_ge_85":False,\n            "note":"Development CV selects the feature set and threshold. The latest filing-year holdout is evaluated once. Development folds are issuer-grouped. The reviewed queue is a curated case-control sample, not natural prevalence."\n                +(f" The archive proxy labels were ignored; TF-IDF/SVD was fit only on filings before {holdout_start_year}."\n                  if archive_meta else "")}\n    # Evaluate every prespecified feature view on the same untouched future years.\n    # Every view\'s model and threshold come from grouped development data only.\n    if not hold.empty and hold.target_restatement.nunique()==2 and hold_bal.target_restatement.nunique()==2:\n        train=dev.copy()\n        lookup=dev.set_index("adsh")["_embed_pos"]\n        train_pos=lookup.loc[train.adsh.astype(str)].to_numpy(dtype=int)\n        hold_pos=frame.index[held_mask]\n        train_weights=_class_weights(train)\n        holdout_probabilities={}\n        prediction_frames=[]\n        output_dir=Path(output_dir); output_dir.mkdir(parents=True,exist_ok=True)\n        for view_index,(model_name,selected) in enumerate(best_by_config.items()):\n            engine,_,depth,trees=selected["key"]\n            view=selected["key"][1]\n            view_threshold=selected["threshold"]\n            xtr,xh,feature_names=_features(train,hold,view,embeds,train_pos,hold_pos,\n                                           return_names=True)\n            if engine=="xgboost":\n                final=xgb.XGBClassifier(n_estimators=trees,max_depth=depth,learning_rate=.035,\n                    min_child_weight=4,subsample=.85,colsample_bytree=.85,reg_lambda=8,\n                    objective="binary:logistic",eval_metric="logloss",tree_method="hist",device="cuda",\n                    random_state=seed+view_index,n_jobs=2)\n                final.fit(xtr,train.target_restatement.astype(int),sample_weight=train_weights)\n                _assert_cuda_booster(final)\n                model_path=output_dir/f"model_{view}_{engine}.json"\n            else:\n                final=CatBoostClassifier(iterations=trees,depth=depth,learning_rate=.035,\n                    loss_function="Logloss",eval_metric="Logloss",task_type="GPU",devices="0",\n                    l2_leaf_reg=8,random_seed=seed+view_index,verbose=False,allow_writing_files=False)\n                final.fit(xtr,train.target_restatement.astype(int),sample_weight=train_weights,\n                          verbose=False)\n                _assert_catboost_gpu(final)\n                model_path=output_dir/f"model_{view}_{engine}.cbm"\n            hold_probs=final.predict_proba(xh)[:,1]\n            holdout_probabilities[model_name]=hold_probs\n            prob_lookup=dict(zip(hold.adsh.astype(str),hold_probs))\n            balanced_prob=np.asarray([prob_lookup[a] for a in hold_bal.adsh.astype(str)])\n            balanced_report=_report(hold_bal.target_restatement,balanced_prob,view_threshold)\n            all_report=_report(hold.target_restatement,hold_probs,view_threshold)\n            result["holdout_results_by_view"][model_name]={\n                "engine":engine,"view":view,"threshold":view_threshold,\n                "balanced_holdout_1_to_1":balanced_report,\n                "balanced_holdout_1_to_1_ci95":_bootstrap_metric_intervals(\n                    hold_bal.target_restatement,balanced_prob,view_threshold,seed+3000+view_index),\n                "curated_holdout_all_rows":all_report,\n                "curated_holdout_all_rows_ci95":_bootstrap_metric_intervals(\n                    hold.target_restatement,hold_probs,view_threshold,seed+4000+view_index)}\n            prediction_frames.append(pd.DataFrame({\n                "adsh":hold.adsh.astype(str),target_column:hold.target_restatement.astype(int),\n                "model_view":view,"model_engine":engine,"threshold":view_threshold,\n                "probability":hold_probs,"predicted":(hold_probs>=view_threshold).astype(int),\n                "cik_group_hash":[hashlib.sha256(str(v).encode()).hexdigest()[:16]\n                                  for v in hold.cik_norm]}))\n            final.save_model(model_path)\n            if selected["key"]==best_key:\n                result["balanced_holdout_1_to_1"]=balanced_report\n                result["balanced_holdout_1_to_1_ci95"]=result["holdout_results_by_view"][model_name]["balanced_holdout_1_to_1_ci95"]\n                result["curated_holdout_all_rows"]=all_report\n                result["curated_holdout_all_rows_ci95"]=result["holdout_results_by_view"][model_name]["curated_holdout_all_rows_ci95"]\n                result["metric_goal_all_four_ge_85"]=all(balanced_report[k]>=.85 for k in\n                    ("accuracy","precision","recall","f1"))\n                sensitivity=[]\n                for draw in range(50):\n                    sample=_balance(hold,seed+2000+draw)\n                    if sample.empty or sample.target_restatement.nunique()<2:\n                        continue\n                    sampled_prob=np.asarray([prob_lookup[a] for a in sample.adsh.astype(str)])\n                    sensitivity.append(_report(sample.target_restatement,sampled_prob,view_threshold))\n                report_metrics=("accuracy","precision","recall","f1","roc_auc","pr_auc")\n                result["matched_sample_sensitivity_50"]={\n                    "draws":len(sensitivity),\n                    "metrics":{name:{"median":float(np.nanmedian([row[name] for row in sensitivity if row[name] is not None])),\n                                      "p05":float(np.nanquantile([row[name] for row in sensitivity if row[name] is not None],.05)),\n                                      "p95":float(np.nanquantile([row[name] for row in sensitivity if row[name] is not None],.95))}\n                               for name in report_metrics if any(row[name] is not None for row in sensitivity)},\n                    "fraction_all_four_ge_85":float(np.mean([all(row[k]>=.85 for k in ("accuracy","precision","recall","f1"))\n                                                              for row in sensitivity])) if sensitivity else None,\n                    "note":"Repeated 1:1 matching sensitivity, not a confidence interval."}\n                try:\n                    _save_shap_importance(final,engine,xh,hold.target_restatement,\n                                          feature_names,output_dir)\n                    result["shap_explanations"]="feature_importance_shap.csv and feature_group_importance_shap.csv"\n                except Exception as exc:\n                    result["shap_explanations_error"]=f"{type(exc).__name__}: {exc}"[:500]\n        result["paired_holdout_auc_differences"]={}\n        for engine in ("xgboost","catboost"):\n            if f"{engine}/financial" not in holdout_probabilities:\n                continue\n            baseline=holdout_probabilities[f"{engine}/financial"]\n            baseline_balanced=np.asarray([dict(zip(hold.adsh.astype(str),baseline))[a]\n                                          for a in hold_bal.adsh.astype(str)])\n            for view_index,view in enumerate(views):\n                if view=="financial":\n                    continue\n                model_name=f"{engine}/{view}"\n                if model_name not in holdout_probabilities:\n                    continue\n                view_prob=holdout_probabilities[model_name]\n                view_balanced=np.asarray([dict(zip(hold.adsh.astype(str),view_prob))[a]\n                                          for a in hold_bal.adsh.astype(str)])\n                result["paired_holdout_auc_differences"][model_name]={\n                    "balanced_holdout_1_to_1":_paired_auc_difference(\n                        hold_bal.target_restatement,view_balanced,baseline_balanced,\n                        seed+5000+view_index),\n                    "curated_holdout_all_rows":_paired_auc_difference(\n                        hold.target_restatement,view_prob,baseline,seed+6000+view_index)}\n        pd.concat(prediction_frames,ignore_index=True).to_csv(\n            output_dir/"future_holdout_predictions.csv",index=False)\n    output_dir=Path(output_dir); output_dir.mkdir(parents=True,exist_ok=True)\n    leaderboard.to_csv(output_dir/"grouped_cv_leaderboard.csv",index=False)\n    (output_dir/"metrics.json").write_text(json.dumps(result,indent=2,default=str),encoding="utf-8")\n    def pct(value):\n        return "NA" if value is None else f"{100*float(value):.1f}%"\n    balanced=result.get("balanced_holdout_1_to_1") or {}\n    all_rows=result.get("curated_holdout_all_rows") or {}\n    holdout_views=result.get("holdout_results_by_view",{})\n    balanced_rows=[]\n    all_rows_lines=[]\n    for view,view_result in holdout_views.items():\n        for metric_key,destination in (("balanced_holdout_1_to_1",balanced_rows),\n                                       ("curated_holdout_all_rows",all_rows_lines)):\n            metrics=view_result.get(metric_key,{})\n            destination.append(\n                f"| {view} | {metrics.get(\'n\',0)} | {pct(metrics.get(\'accuracy\'))} | "\n                f"{pct(metrics.get(\'precision\'))} | {pct(metrics.get(\'recall\'))} | "\n                f"{pct(metrics.get(\'f1\'))} | {pct(metrics.get(\'roc_auc\'))} | "\n                f"{pct(metrics.get(\'pr_auc\'))} |")\n    paired_lines=[]\n    for view,comparison in result.get("paired_holdout_auc_differences",{}).items():\n        values=comparison.get("balanced_holdout_1_to_1",{})\n        pr=values.get("pr_auc_difference",{})\n        roc=values.get("roc_auc_difference",{})\n        paired_lines.append(f"- `{view}` minus financial baseline, matched holdout: "\n            f"PR-AUC difference {pct(pr.get(\'estimate\'))} "\n            f"(95% CI {pct(pr.get(\'lower_95\'))} to {pct(pr.get(\'upper_95\'))}); "\n            f"ROC-AUC difference {pct(roc.get(\'estimate\'))} "\n            f"(95% CI {pct(roc.get(\'lower_95\'))} to {pct(roc.get(\'upper_95\'))}).")\n    report=(f"# MD&A outcome study: {target_column}\\n\\n"\n            f"**Outcome definition:** {target_definition}\\n\\n"\n            "The evaluation uses issuer-grouped development folds and a later-year holdout. "\n            "All reported metrics refer to the named outcome above.\\n\\n"\n            f"- Reviewed model rows: {result.get(\'rows\', 0)} "\n            f"({result.get(\'positives\', 0)} positive; {result.get(\'negatives\', 0)} negative).\\n"\n            f"- Forward holdout begins in {result.get(\'future_holdout_start_year\', \'NA\')}; "\n            f"{result.get(\'holdout_rows\', 0)} holdout rows before 1:1 matching.\\n"\n            + (f"- Full archive text features: {result[\'full_archive_text_features\'].get(\'archive_rows\', 0)} documents; "\n               f"{result[\'full_archive_text_features\'].get(\'archive_fit_rows\', 0)} pre-holdout documents fit the text transform "\n               f"through {result[\'full_archive_text_features\'].get(\'fit_through_year\', \'NA\')}; legacy proxy labels were ignored.\\n"\n               if result.get(\'full_archive_text_features\') else "")\n            + f"- Selected model: {result.get(\'selected_model\', {}).get(\'engine\', \'NA\')} / "\n            f"{result.get(\'selected_model\', {}).get(\'view\', \'NA\')}; selected using issuer-grouped development PR-AUC.\\n\\n"\n            "## Forward holdout: year/form-matched 1:1\\n\\n"\n            "| Feature view | n | Accuracy | Precision | Recall | F1 | ROC-AUC | PR-AUC |\\n"\n            "|---|---:|---:|---:|---:|---:|---:|---:|\\n"\n            + "\\n".join(balanced_rows) + "\\n\\n"\n            "## Forward holdout: all curated rows\\n\\n"\n            "| Feature view | n | Accuracy | Precision | Recall | F1 | ROC-AUC | PR-AUC |\\n"\n            "|---|---:|---:|---:|---:|---:|---:|---:|\\n"\n            + "\\n".join(all_rows_lines) + "\\n\\n"\n            "## Paired feature-view differences\\n\\n"\n            + ("\\n".join(paired_lines) if paired_lines else "No paired comparison was produced.") + "\\n\\n"\n            "Bootstrap 95% intervals are in `metrics.json`. Feature attributions are in the SHAP CSV files.\\n\\n"\n            "## Limits\\n\\n"\n            "The source cohort is enriched and sampled. The 1:1 results apply only to the matched case-control "\n            "sample, and precision does not estimate population precision. If the target uses AI-assisted "\n            "outcome screening, report that labeling method and its uncertainty.\\n")\n    (output_dir/"study_report.md").write_text(report,encoding="utf-8")\n    return result\n\n'}
for name,source in SOURCES.items(): (MODULES/name).write_text(source,encoding="utf-8")
if str(MODULES) not in sys.path: sys.path.insert(0,str(MODULES))
import research_v5_core as core; import sec_v5_evidence as evidence
importlib.reload(core); importlib.reload(evidence)
spec=importlib.util.spec_from_file_location("restatement_study",MODULES/"22_restatement_study.py")
study=importlib.util.module_from_spec(spec); spec.loader.exec_module(study)
print("Embedded research modules loaded.")

In [ ]:
import pandas as pd, getpass
ledger=pd.read_csv(LEDGER,dtype={"original_adsh":str},low_memory=False)
errors=study.validate_ledger(ledger)
if errors: raise ValueError("Ledger errors:\n- "+"\n- ".join(errors))
reviewed=ledger.review_status.fillna("").astype(str).str.lower().eq("reviewed").sum()
print(f"Queue rows: {len(ledger):,}; reviewed: {reviewed:,}; pending: {len(ledger)-reviewed:,}")
if not reviewed: raise ValueError("No outcomes are marked reviewed yet. Prepare an evidence-reviewed ledger first; proxy labels cannot substitute.")
proxy=pd.read_csv(PROXY,dtype={"adsh":str,"cik":str},low_memory=False)
USER_AGENT=getpass.getpass("Real SEC User-Agent (name/organization and contact email): ").strip()
MAX_NEW_PER_SESSION=500  # Resume this cell in later sessions until pending reaches 0; use 0 for all remaining.
report=study.collect_reviewed_mda(proxy,ledger,HORIZON_YEARS,CACHE,USER_AGENT,max_new=MAX_NEW_PER_SESSION)
print(report)

## Prepare data and train

If the preceding cell reports `pending` above zero, rerun it in a later session until all current and prior MD&A sections are cached. Then run the next cell. It retains every horizon-eligible reviewed row, records text coverage, and never adds labels to pending rows.

In [ ]:
if report["pending"]: raise RuntimeError("SEC text collection is incomplete. Rerun the collection cell until pending is 0.")
DATASET=ROOT/f"data/confirmed_restatement_h{HORIZON_YEARS}y_fulltext_trajectory_v2.csv.gz"
prepared=study.prepare_model_data(PROXY,LEDGER,CACHE,HORIZON_YEARS,DATASET)
print(prepared)
counts=pd.read_csv(DATASET,compression="gzip",usecols=["target_restatement"]).target_restatement.value_counts().to_dict()
print("Usable target counts:",counts)

In [ ]:
metrics=study.train_gpu(DATASET,OUTPUT)
print(json.dumps(metrics,indent=2))
print("\nOutput folder:",OUTPUT)
print("The run saves six model/view evaluations, their model files, study_report.md, and SHAP summaries when available.")

## Research question and reporting

**Suggested title:** *Does MD&A change add predictive value for a later SEC-reported restatement? A three-year matched temporal-holdout study.*

**Primary question:** Do current MD&A language and its change from the prior comparable filing improve prediction beyond structured financial features? The three prespecified views are evaluated on the same forward holdout, with each model and threshold selected using development data only. The main incremental comparison is the paired holdout PR-AUC difference between the financial-plus-text view and the financial baseline.

Report the financial-only, interpretable-language, and semantic-change comparisons. Use `grouped_cv_leaderboard.csv` for development model selection; the frozen forward holdout is reported once. `feature_importance_shap.csv` lists feature attributions, and `feature_group_importance_shap.csv` summarizes them by financial, lexical, embedding, and availability groups. These explain the fitted model; they do not establish causation.

## Reading the result

`balanced_holdout_1_to_1` is the primary matched-sample metric set. It is evaluated on the latest filing years, which were excluded from training and threshold selection, with year/form-matched 1:1 sampling. Its `*_ci95` fields are stratified bootstrap intervals for that sample. Companies can appear in earlier development years; this reflects prediction for continuing issuers, while development folds group issuers. `holdout_adequate_for_claim` requires at least 30 cases per class and 10 distinct issuers per class. If false, treat the metrics as preliminary. `matched_sample_sensitivity_50` shows how the result changes across 50 fixed-threshold 1:1 matches; it is a matching sensitivity analysis, not a confidence interval. `development_cv_1_to_1` selects the model and threshold; do not present it as a final estimate. The curated all-row holdout is not a natural-prevalence estimate because the review queue was sampled. `study_report.md` assembles counts, results, and limitations for the report draft.

If the GPU cell says there are too few positive/negative rows or issuer groups, expand the reviewed ledger first. Do not balance by duplicating rows, include proxy labels as target, or tune on the held-out results. After selecting a horizon and model using development evidence, collect new mature filings for a future external test.